# Complete Video Music Pipeline (Self-Contained for Kaggle)

**Pipeline Steps:**
1. Video Analysis - Speech-to-text (Whisper) + frame-by-frame visuals
1.5. Celebrity & Music Director Analysis - Detect heroes/heroines, find their hit music directors
2. BGM Prompt Generation - AWS Bedrock (Claude) with preferred music director style
3. Music Generation - MusicGen
4. Audio Mixing - Voice clear + BGM subtle
5. Final Video - Output with synced BGM

In [1]:
## 0. Install Dependencies

In [1]:
!pip install -q torch torchvision torchaudio
!pip install -q transformers scipy scikit-learn
!pip install -q openai-whisper
!pip install -q opencv-python-headless
!pip install -q boto3 Pillow

# Clear any corrupted Hugging Face cache (helps with Kaggle connectivity issues)
import os
import shutil
hf_cache = os.path.expanduser('~/.cache/huggingface')
if os.path.exists(hf_cache):
    print('Clearing Hugging Face cache...')
    shutil.rmtree(hf_cache, ignore_errors=True)
    print('Cache cleared!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 11.2 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 1. Imports & Configuration

In [2]:
import os, json, subprocess, warnings, tempfile, glob, platform, re
import numpy as np
import cv2
import torch
import whisper
import boto3
import scipy.io.wavfile
from pathlib import Path
from typing import Optional, Dict, List, Any
from collections import Counter
from PIL import Image
from transformers import (
    CLIPProcessor, CLIPModel,
    pipeline as hf_pipeline,
    DetrImageProcessor, DetrForObjectDetection,
    BlipProcessor, BlipForConditionalGeneration,
    AutoProcessor, MusicgenForConditionalGeneration
)
from sklearn.cluster import KMeans
warnings.filterwarnings('ignore')

# ── USER CONFIG ─────────────────────────────────────────────────
VIDEO_PATH = "/kaggle/input/datasets/nagadeepak612005/pradeep-ranganathan2-0/12142366-uhd_4096_2160_24fps_with_bgm (online-video-cutter.com) (1).mp4"  # <-- SET THIS
OUTPUT_DIR = "/kaggle/working/output"
FPS = 1
BGM_VOLUME = 0.40
WHISPER_MODEL = "base"
MUSIC_MODEL = "facebook/musicgen-small"
USE_BEDROCK = True

# ── AWS CONFIG ──────────────────────────────────────────────────
#place your config here

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
video_name = Path(VIDEO_PATH).stem
print(f"Video: {VIDEO_PATH}")
print(f"Output: {OUTPUT_DIR}")

Video: /kaggle/input/datasets/nagadeepak612005/pradeep-ranganathan2-0/12142366-uhd_4096_2160_24fps_with_bgm (online-video-cutter.com) (1).mp4
Output: /kaggle/working/output


## 2. Helper: convert_to_native

In [3]:
def convert_to_native(obj):
    """Convert numpy types to native Python types"""
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {key: convert_to_native(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_native(item) for item in obj]
    return obj

print("Helper loaded.")

Helper loaded.


## 3. CompleteVideoAnalyzer

In [5]:
class CompleteVideoAnalyzer:
    """Complete video analysis: speech-to-text + frame-by-frame visual analysis."""

    def __init__(self, whisper_model: str = "base"):
        print("=" * 70)
        print("INITIALIZING COMPLETE VIDEO ANALYZER")
        print("=" * 70)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {self.device}\n")

        print("Loading Whisper model...")
        self.whisper_model = whisper.load_model(whisper_model, device=self.device)
        print("✓ Whisper loaded\n")

        print("Loading CLIP model...")
        # Use local cache directory to avoid conflicts in Kaggle
        clip_cache_dir = "/kaggle/working/clip_cache"
        os.makedirs(clip_cache_dir, exist_ok=True)
        
        try:
            # Try loading with explicit cache and resume_download
            self.clip_processor = CLIPProcessor.from_pretrained(
                "openai/clip-vit-base-patch32",
                cache_dir=clip_cache_dir,
                resume_download=True
            )
            self.clip_model = CLIPModel.from_pretrained(
                "openai/clip-vit-base-patch32",
                cache_dir=clip_cache_dir,
                resume_download=True
            ).to(self.device)
        except Exception as e:
            print(f"Failed with openai/clip-vit-base-patch32: {e}")
            print("Trying alternative: openai/clip-vit-base-patch16...")
            # Fallback to patch16 which is more commonly available
            self.clip_processor = CLIPProcessor.from_pretrained(
                "openai/clip-vit-base-patch16",
                cache_dir=clip_cache_dir
            )
            self.clip_model = CLIPModel.from_pretrained(
                "openai/clip-vit-base-patch16",
                cache_dir=clip_cache_dir
            ).to(self.device)
        print("✓ CLIP loaded\n")

        print("Loading BLIP captioning model...")
        self.caption_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
        self.caption_model = BlipForConditionalGeneration.from_pretrained(
            "Salesforce/blip-image-captioning-base"
        ).to(self.device)
        print("✓ BLIP loaded\n")

        print("Loading emotion recognition model...")
        self.emotion_classifier = hf_pipeline(
            "image-classification",
            model="dima806/facial_emotions_image_detection",
            device=0 if self.device == "cuda" else -1
        )
        print("✓ Emotion classifier loaded\n")

        print("Loading object detection model...")
        self.object_processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
        self.object_model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50").to(self.device)
        print("✓ Object detector loaded\n")
        print("ALL MODELS LOADED!\n")

    def transcribe_video_audio(self, video_path: str, language=None) -> Dict:
        print("STEP 1: AUDIO TRANSCRIPTION")
        try:
            result = self.whisper_model.transcribe(video_path, language=language, task="transcribe", verbose=False)
            detected_language = result.get("language", "unknown")
            print(f"✓ Detected language: {detected_language}")
            return {
                "text": result["text"].strip(),
                "language": detected_language,
                "segments": [{"start": s["start"], "end": s["end"], "text": s["text"].strip()} for s in result["segments"]],
                "duration": result["segments"][-1]["end"] if result["segments"] else 0,
            }
        except Exception as e:
            print(f"⚠ Audio transcription failed: {e}")
            return {"text": "", "language": "none", "segments": [], "duration": 0, "error": str(e)}

    def generate_caption(self, image: np.ndarray) -> str:
        pil_image = Image.fromarray(image)
        inputs = self.caption_processor(pil_image, return_tensors="pt").to(self.device)
        with torch.no_grad():
            out = self.caption_model.generate(**inputs, max_length=50)
        return self.caption_processor.decode(out[0], skip_special_tokens=True)

    def extract_descriptors(self, caption: str) -> List[str]:
        stop_words = {"a", "an", "the", "is", "are", "in", "on", "at", "to", "of", "and", "or"}
        return [w.strip(".,!?") for w in caption.lower().split() if w not in stop_words and len(w) > 2][:5]

    def query_with_clip(self, image: np.ndarray, concepts: List[str]) -> List[Dict]:
        if not concepts:
            return []
        pil_image = Image.fromarray(image)
        inputs = self.clip_processor(text=concepts, images=pil_image, return_tensors="pt", padding=True).to(self.device)
        with torch.no_grad():
            outputs = self.clip_model(**inputs)
            probs = outputs.logits_per_image.softmax(dim=1)
        top_k = min(3, len(concepts))
        top_probs, top_indices = torch.topk(probs[0], top_k)
        return [{"type": concepts[idx], "confidence": round(prob.item(), 3)} for prob, idx in zip(top_probs, top_indices)]

    def analyze_scene(self, image: np.ndarray, caption: str) -> List[Dict]:
        descriptors = self.extract_descriptors(caption)
        queries = list(set([q for w in descriptors for q in [w, f"{w} scene", f"{w} environment"]]))
        return self.query_with_clip(image, queries) if queries else [{"type": caption, "confidence": 1.0}]

    def analyze_mood(self, image: np.ndarray, caption: str) -> List[Dict]:
        return self.query_with_clip(image, [f"a {caption}", f"{caption} with emotional atmosphere", f"mood of {caption}"])

    def detect_objects(self, image: np.ndarray) -> List[Dict]:
        pil_image = Image.fromarray(image)
        inputs = self.object_processor(images=pil_image, return_tensors="pt").to(self.device)
        with torch.no_grad():
            outputs = self.object_model(**inputs)
        target_sizes = torch.tensor([pil_image.size[::-1]]).to(self.device)
        results = self.object_processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=0.3)[0]
        return [
            {"object": self.object_model.config.id2label[label.item()], "confidence": round(score.item(), 3), "bbox": [round(c, 2) for c in box.tolist()]}
            for score, label, box in zip(results["scores"], results["labels"], results["boxes"])
        ]

    def detect_emotions(self, image: np.ndarray) -> List[Dict]:
        try:
            results = self.emotion_classifier(Image.fromarray(image), top_k=3)
            return [{"emotion": r["label"], "confidence": round(r["score"], 3)} for r in results]
        except:
            return []

    def analyze_colors(self, image: np.ndarray) -> Dict:
        pixels = cv2.resize(image, (150, 150)).reshape(-1, 3)
        kmeans = KMeans(n_clusters=5, random_state=42, n_init=10).fit(pixels)
        colors = kmeans.cluster_centers_.astype(int)
        counts = np.bincount(kmeans.labels_)
        sorted_idx = np.argsort(counts)[::-1]
        dominant_colors = []
        for idx in sorted_idx:
            rgb = colors[idx].tolist()
            dominant_colors.append({"rgb": rgb, "hex": "#{:02x}{:02x}{:02x}".format(*rgb), "percentage": round((counts[idx] / len(pixels)) * 100, 2)})
        avg = np.mean(pixels, axis=0)
        r, g, b = avg
        temperature = "warm" if r > b + 20 else "cool" if b > r + 20 else "neutral"
        brightness = round(np.mean(avg) / 255.0, 2)
        return {"dominant_colors": dominant_colors, "temperature": temperature, "brightness": brightness, "brightness_level": "bright" if brightness > 0.6 else "dark" if brightness < 0.3 else "medium"}

    def calculate_motion(self, prev_frame, curr_frame) -> float:
        if prev_frame is None:
            return 0.0
        prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_RGB2GRAY)
        curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_RGB2GRAY)
        flow = cv2.calcOpticalFlowFarneback(prev_gray, curr_gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        return float(np.mean(np.sqrt(flow[..., 0] ** 2 + flow[..., 1] ** 2)))

    def extract_frames(self, video_path: str, fps: int = 1) -> List[Dict]:
        print("STEP 2: FRAME EXTRACTION")
        cap = cv2.VideoCapture(video_path)
        video_fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration = total_frames / video_fps
        print(f"Video: {duration:.1f}s, {video_fps:.1f} FPS, extracting at {fps} FPS")
        frames, frame_interval, frame_count, prev_frame = [], int(video_fps / fps), 0, None
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if frame_count % frame_interval == 0:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                motion = self.calculate_motion(prev_frame, frame_rgb)
                frames.append({"frame_number": frame_count, "timestamp": frame_count / video_fps, "image": frame_rgb, "motion_intensity": motion})
                prev_frame = frame_rgb.copy()
            frame_count += 1
        cap.release()
        print(f"✓ Extracted {len(frames)} frames\n")
        return frames

    def analyze_frame(self, frame_data: Dict) -> Dict:
        image = frame_data["image"]
        caption = self.generate_caption(image)
        descriptors = self.extract_descriptors(caption)
        return {
            "timestamp": round(frame_data["timestamp"], 2),
            "frame_number": frame_data["frame_number"],
            "caption": caption,
            "descriptors": descriptors,
            "objects": self.detect_objects(image),
            "emotions": self.detect_emotions(image),
            "facial_expressions": self.detect_emotions(image),
            "colors": self.analyze_colors(image),
            "scene": self.analyze_scene(image, caption),
            "mood": self.analyze_mood(image, caption),
            "movement": {"intensity": round(frame_data["motion_intensity"], 2), "level": "slow" if frame_data["motion_intensity"] < 2.0 else "moderate" if frame_data["motion_intensity"] < 4.0 else "fast"},
        }

    def generate_summary(self, frame_analyses: List[Dict], transcription: Dict) -> Dict:
        all_objects, all_emotions, all_scenes, all_moods = [], [], [], []
        for f in frame_analyses:
            all_objects.extend([o["object"] for o in f.get("objects", [])])
            all_emotions.extend([e["emotion"] for e in f.get("emotions", [])])
            if f.get("scene"):
                all_scenes.append(f["scene"][0]["type"])
            if f.get("mood"):
                all_moods.append(f["mood"][0]["type"])
        obj_c, emo_c, scene_c, mood_c = Counter(all_objects), Counter(all_emotions), Counter(all_scenes), Counter(all_moods)
        motion_vals = [f["movement"]["intensity"] for f in frame_analyses]
        avg_motion = np.mean(motion_vals) if motion_vals else 0
        temp_c = Counter([f["colors"]["temperature"] for f in frame_analyses])
        return {
            "duration_seconds": round(frame_analyses[-1]["timestamp"], 1) if frame_analyses else 0,
            "total_frames_analyzed": len(frame_analyses),
            "has_audio": bool(transcription.get("text")),
            "transcription_summary": {"language": transcription.get("language", "none"), "total_text_length": len(transcription.get("text", "")), "segment_count": len(transcription.get("segments", []))},
            "common_objects": [{"object": k, "count": v} for k, v in obj_c.most_common(10)],
            "detected_emotions": [{"emotion": k, "count": v} for k, v in emo_c.most_common(5)],
            "dominant_scenes": [{"type": k, "count": v} for k, v in scene_c.most_common(3)],
            "dominant_moods": [{"mood": k, "count": v} for k, v in mood_c.most_common(3)],
            "motion_analysis": {"average_intensity": round(float(avg_motion), 2), "pacing": "slow-paced" if avg_motion < 2.0 else "moderate-paced" if avg_motion < 4.0 else "fast-paced"},
            "color_analysis": {"temperature_distribution": dict(temp_c), "dominant_temperature": temp_c.most_common(1)[0][0] if temp_c else "unknown"},
        }

    def analyze_video(self, video_path: str, fps: int = 1, language=None) -> Dict:
        print("\nCOMPLETE VIDEO ANALYSIS PIPELINE\n")
        transcription = self.transcribe_video_audio(video_path, language)
        frames = self.extract_frames(video_path, fps)
        if not frames:
            return {"error": "No frames extracted"}
        print("STEP 3: FRAME-BY-FRAME ANALYSIS")
        frame_analyses = []
        for i, fd in enumerate(frames):
            print(f"Analyzing frame {i+1}/{len(frames)} at {fd['timestamp']:.2f}s...")
            frame_analyses.append(self.analyze_frame(fd))
        print("✓ Frame analysis complete\n")
        print("STEP 4: GENERATING SUMMARY")
        summary = self.generate_summary(frame_analyses, transcription)
        print("✓ Summary generated\n")
        return {"video_path": video_path, "transcription": transcription, "summary": summary, "frame_by_frame_analysis": frame_analyses}

print("CompleteVideoAnalyzer defined.")


CompleteVideoAnalyzer defined.


## 3.5 CelebrityMusicDirectorAnalyzer

Detects heroes/heroines/celebrities in video frames using face crops + Bedrock LLM.
Looks up their filmography and identifies the music director who gave them the most hit songs.
This preferred music director is then used as a style reference for BGM generation.

In [9]:
import requests
import json as _json
from urllib.parse import quote as _quote
from collections import Counter as _Counter

class CelebrityMusicDirectorAnalyzer:
    """
    Detect celebrities/actors in video frames and find their most
    frequent hit music director using Bedrock LLM.
    
    Flow:
      1. Extract face crops from video frames (using OpenCV Haar cascade)
      2. Send face crops + video context to Bedrock LLM
      3. LLM identifies celebrities, their filmography, and top music directors
      4. Returns preferred music director(s) as style references
    """

    def __init__(self):
        print("Initializing CelebrityMusicDirectorAnalyzer...")
        self.api_key = BEDROCK_API_KEY
        self.model_id = BEDROCK_MODEL_ID
        self.region = AWS_REGION
        encoded_model_id = _quote(self.model_id, safe="")
        self.endpoint = (
            f"https://bedrock-runtime.{self.region}.amazonaws.com"
            f"/model/{encoded_model_id}/invoke"
        )
        # Load face detector
        cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
        self.face_cascade = cv2.CascadeClassifier(cascade_path)
        print("✓ CelebrityMusicDirectorAnalyzer ready\n")

    def _invoke_model(self, body: dict) -> str:
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
            "Accept": "application/json",
        }
        response = requests.post(self.endpoint, headers=headers, json=body, timeout=120)
        if response.status_code != 200:
            raise ValueError(f"Bedrock API error {response.status_code}: {response.text[:300]}")
        result = response.json()
        if "content" in result:
            return result["content"][0]["text"]
        elif "body" in result:
            inner = _json.loads(result["body"]) if isinstance(result["body"], str) else result["body"]
            return inner["content"][0]["text"]
        raise ValueError(f"Unexpected response: {list(result.keys())}")

    def extract_face_crops(self, frames, max_faces=15):
        """Extract unique face crops from video frames."""
        print("  Extracting face crops from frames...")
        all_crops = []
        for i, frame_data in enumerate(frames):
            if isinstance(frame_data, dict):
                image = frame_data.get("image")
                if image is None:
                    continue
            else:
                image = frame_data

            gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
            faces = self.face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))

            for (x, y, w, h) in faces:
                # Add padding around face
                pad = int(0.3 * w)
                x1 = max(0, x - pad)
                y1 = max(0, y - pad)
                x2 = min(image.shape[1], x + w + pad)
                y2 = min(image.shape[0], y + h + pad)
                crop = image[y1:y2, x1:x2]
                if crop.shape[0] > 50 and crop.shape[1] > 50:
                    all_crops.append({
                        "crop": crop,
                        "frame_idx": i,
                        "timestamp": frame_data.get("timestamp", i) if isinstance(frame_data, dict) else i,
                        "size": w * h
                    })

        if not all_crops:
            print("  No faces detected in video frames.")
            return []

        # Sort by size (larger faces = more prominent) and deduplicate
        all_crops.sort(key=lambda x: x["size"], reverse=True)
        selected = all_crops[:max_faces]
        print(f"  ✓ Found {len(all_crops)} face detections, selected top {len(selected)}\n")
        return selected

    def analyze_celebrities(self, analysis: dict) -> dict:
        """
        Main method: detect celebrities in video and find their preferred music directors.
        
        Returns dict with:
          - detected_celebrities: list of identified persons
          - preferred_music_directors: list of music directors ranked by relevance
          - filmography_summary: brief summary of actor-music director connections
        """
        print("=" * 70)
        print("CELEBRITY & MUSIC DIRECTOR ANALYSIS")
        print("=" * 70)

        frames = analysis.get("frame_by_frame_analysis", [])
        transcription = analysis.get("transcription", {})
        summary = analysis.get("summary", {})

        # Gather context clues from the video
        language = transcription.get("language", "unknown")
        text_snippet = transcription.get("text", "")[:500]
        captions = [f.get("caption", "") for f in frames[:10]]
        objects = [o["object"] for o in summary.get("common_objects", [])[:10]]
        emotions = [e["emotion"] for e in summary.get("detected_emotions", [])[:5]]

        # Count person detections across frames
        person_count = sum(1 for f in frames for o in f.get("objects", []) if o.get("object") == "person")

        if person_count < 2:
            print("  Very few person detections — skipping celebrity analysis.")
            return self._empty_result()

        # Build the LLM prompt with all available context
        system_prompt = """You are an expert in identifying celebrities, actors, heroines, and important public figures 
from video content. You also have deep knowledge of film industries worldwide — especially 
Indian cinema (Tamil, Telugu, Hindi, Malayalam, Kannada), Hollywood, Korean, Japanese, and others.

Your task:
1. Based on the video context (language, speech transcript, scene descriptions, objects detected), 
   identify if any known actors/heroines/celebrities are likely present.
2. For each identified person, list their notable films (top 5-10).
3. For each person, identify which MUSIC DIRECTOR they have worked with most frequently 
   and who composed the most hit/iconic songs for their films.
4. Rank the music directors by how many hit collaborations they had with the detected celebrities.

OUTPUT STRICT JSON:
{
  "detected_celebrities": [
    {
      "name": "<full name>",
      "role": "<hero/heroine/director/singer/public figure>",
      "confidence": "<high/medium/low>",
      "detection_reason": "<why you think this person is in the video>",
      "notable_films": ["<film1>", "<film2>", ...],
      "top_music_directors": [
        {"name": "<music director>", "hit_films_together": ["<film1>", "<film2>"], "hit_count": <number>}
      ]
    }
  ],
  "preferred_music_directors": [
    {
      "name": "<music director name>",
      "total_hits_with_detected_celebrities": <number>,
      "style_description": "<brief description of their musical style>",
      "signature_instruments": ["<instrument1>", "<instrument2>"],
      "notable_works": ["<song/film1>", "<song/film2>"]
    }
  ],
  "filmography_summary": "<2-3 sentence summary of the actor-music director connections found>"
}

RULES:
- If you cannot identify any specific celebrity, still analyze the cultural context and suggest 
  music directors who are iconic for that language/region's film industry.
- Be honest about confidence levels.
- Output ONLY valid JSON, no markdown, no extra text.
- Focus on music directors who are MOST RELEVANT to the detected celebrities."""

        user_message = f"""Analyze this video content and identify any celebrities/actors/heroines present.
Then find their most frequent and hit-giving music directors.

VIDEO CONTEXT:
- Detected language: {language}
- Person detections across frames: {person_count}
- Speech transcript (first 500 chars): {text_snippet}

SCENE DESCRIPTIONS (from frame captions):
{chr(10).join(f'  - {c}' for c in captions if c)}

OBJECTS DETECTED: {', '.join(objects)}
EMOTIONS DETECTED: {', '.join(emotions)}

Based on all this context — especially the language, transcript content, and visual descriptions — 
identify who the celebrities/actors might be and find their preferred music directors."""

        try:
            print("  Sending video context to Bedrock for celebrity identification...")
            raw_text = self._invoke_model({
                "anthropic_version": "bedrock-2023-05-31",
                "max_tokens": 2000,
                "temperature": 0.3,
                "system": system_prompt,
                "messages": [{"role": "user", "content": user_message}],
            })

            # Parse JSON response
            start = raw_text.find("{")
            end = raw_text.rfind("}") + 1
            if start != -1 and end > start:
                result = _json.loads(raw_text[start:end])
            else:
                raise ValueError("No JSON found in response")

            # Print results
            celebrities = result.get("detected_celebrities", [])
            preferred_mds = result.get("preferred_music_directors", [])

            print(f"\n  Detected Celebrities: {len(celebrities)}")
            for c in celebrities:
                print(f"    - {c['name']} ({c.get('role', 'unknown')}) [confidence: {c.get('confidence', 'N/A')}]")
                top_mds = c.get("top_music_directors", [])
                for md in top_mds[:2]:
                    print(f"      → Music Director: {md['name']} ({md.get('hit_count', '?')} hits)")

            print(f"\n  Preferred Music Directors (ranked):")
            for md in preferred_mds:
                print(f"    - {md['name']}: {md.get('total_hits_with_detected_celebrities', '?')} hits")
                print(f"      Style: {md.get('style_description', 'N/A')}")

            print(f"\n  Summary: {result.get('filmography_summary', 'N/A')}\n")
            return result

        except Exception as e:
            print(f"  ⚠ Celebrity analysis failed: {e}")
            print(f"  Falling back to language-based music director suggestion.\n")
            return self._fallback_by_language(language)

    def _empty_result(self):
        return {
            "detected_celebrities": [],
            "preferred_music_directors": [],
            "filmography_summary": "No celebrities detected in the video."
        }

    def _fallback_by_language(self, language):
        """Suggest iconic music directors based on detected language."""
        lang_music_directors = {
            "tamil": [
                {"name": "Ilaiyaraaja", "total_hits_with_detected_celebrities": 0,
                 "style_description": "Carnatic-Western fusion, orchestral mastery, folk roots",
                 "signature_instruments": ["veena", "flute", "violin", "mridangam"],
                 "notable_works": ["Roja Kaadhal Rojave", "Puthiya Mugam", "Nayagan BGM"]},
                {"name": "A.R. Rahman", "total_hits_with_detected_celebrities": 0,
                 "style_description": "Electronic-Carnatic fusion, ambient textures, global sound",
                 "signature_instruments": ["synthesizer", "flute", "tabla", "strings"],
                 "notable_works": ["Roja", "Bombay", "Dil Se"]},
                {"name": "Anirudh Ravichander", "total_hits_with_detected_celebrities": 0,
                 "style_description": "Modern Tamil pop-folk fusion, energetic beats, catchy hooks",
                 "signature_instruments": ["synth bass", "electric guitar", "folk drums"],
                 "notable_works": ["Kolaveri Di", "Master BGM", "Vikram Pathala Pathala"]},
            ],
            "telugu": [
                {"name": "M.M. Keeravani", "total_hits_with_detected_celebrities": 0,
                 "style_description": "Grand orchestral, Carnatic classical, emotional depth",
                 "signature_instruments": ["veena", "sitar", "orchestra", "flute"],
                 "notable_works": ["Baahubali", "RRR Naatu Naatu"]},
                {"name": "S.S. Thaman", "total_hits_with_detected_celebrities": 0,
                 "style_description": "High-energy beats, modern production, mass appeal",
                 "signature_instruments": ["synth", "drums", "brass", "electric guitar"],
                 "notable_works": ["Ala Vaikunthapurramuloo", "Bheemla Nayak"]},
            ],
            "hindi": [
                {"name": "A.R. Rahman", "total_hits_with_detected_celebrities": 0,
                 "style_description": "Sufi-electronic fusion, ambient, orchestral",
                 "signature_instruments": ["synthesizer", "tabla", "flute", "strings"],
                 "notable_works": ["Jai Ho", "Tere Bina", "Kun Faya Kun"]},
                {"name": "Pritam", "total_hits_with_detected_celebrities": 0,
                 "style_description": "Pop-rock Bollywood, catchy melodies, guitar-driven",
                 "signature_instruments": ["acoustic guitar", "piano", "drums"],
                 "notable_works": ["Ae Dil Hai Mushkil", "Gerua", "Kesariya"]},
            ],
            "malayalam": [
                {"name": "M. Jayachandran", "total_hits_with_detected_celebrities": 0,
                 "style_description": "Melodic Carnatic-based, romantic, orchestral",
                 "signature_instruments": ["veena", "flute", "violin", "piano"],
                 "notable_works": ["Ennu Ninte Moideen", "Pranayam"]},
            ],
            "korean": [
                {"name": "Jung Jae-il", "total_hits_with_detected_celebrities": 0,
                 "style_description": "Minimalist, atmospheric, classical-modern blend",
                 "signature_instruments": ["piano", "strings", "cello"],
                 "notable_works": ["Parasite OST", "Squid Game OST"]},
            ],
        }
        directors = lang_music_directors.get(language.lower(), [])
        return {
            "detected_celebrities": [],
            "preferred_music_directors": directors,
            "filmography_summary": f"No specific celebrities identified. Suggesting iconic music directors for {language} cinema."
        }


print("CelebrityMusicDirectorAnalyzer defined.")


CelebrityMusicDirectorAnalyzer defined.


## 4. BedrockBGMPromptGenerator

In [11]:
"""
Updated BedrockBGMPromptGenerator class with 2 LLM calls
Uses Bearer token API Key authentication (no AWS_ACCESS_KEY / AWS_SECRET_KEY needed)
Expanded MusicGen prompt rules for wide global scope
Copy this into your notebook to replace the existing class
"""

import requests
import json
from urllib.parse import quote
from typing import Dict

class BedrockBGMPromptGenerator:
    """
    Generate BGM prompts using AWS Bedrock Claude with 2 LLM calls.

    Flow:
      LLM Call #1 -> Deep video analysis -> Structured JSON (music blueprint)
      LLM Call #2 -> JSON blueprint     -> MusicGen-optimized text prompt (max 500 chars)
    """

    def __init__(self):
        print("Initializing Bedrock client with API Key...")
        self.api_key  = BEDROCK_API_KEY
        self.model_id = BEDROCK_MODEL_ID
        self.region   = AWS_REGION

        encoded_model_id = quote(self.model_id, safe="")
        self.endpoint = (
            f"https://bedrock-runtime.{self.region}.amazonaws.com"
            f"/model/{encoded_model_id}/invoke"
        )
        print(f"Connected to Endpoint : {self.endpoint}")
        print(f"Model                 : {self.model_id}\n")

    # -------------------------------------------------------------------------
    # CORE API CALLER - reused by both LLM calls
    # -------------------------------------------------------------------------

    def _invoke_model(self, body: dict) -> str:
        """Send request to Bedrock using Bearer token auth. Returns raw text."""
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type":  "application/json",
            "Accept":        "application/json",
        }
        response = requests.post(
            self.endpoint,
            headers=headers,
            json=body,
            timeout=120
        )
        if response.status_code != 200:
            raise ValueError(
                f"Bedrock API error {response.status_code}: {response.text[:300]}"
            )
        result = response.json()
        if "content" in result:
            return result["content"][0]["text"]
        elif "body" in result:
            inner = json.loads(result["body"]) if isinstance(result["body"], str) else result["body"]
            return inner["content"][0]["text"]
        else:
            raise ValueError(f"Unexpected response format: {list(result.keys())}")

    # -------------------------------------------------------------------------
    # LLM CALL #1 - Video Understanding -> Music Blueprint (JSON)
    # -------------------------------------------------------------------------

    def create_system_prompt(self) -> str:
        return """<system>
  <role>You are a world-class music director specializing in BGM composition for regional and international video content.</role>

  <task>Deeply analyze frame-level video data and produce a precise MUSIC BLUEPRINT in JSON.</task>

  <cultural_music_detection>
    <priority>Detect the cultural context from language, setting, era, and mood then apply the appropriate musical identity.</priority>

    <region name="SOUTH INDIA (Tamil / Telugu / Kannada / Malayalam)">
      <roots>Identify Carnatic roots, ragas, talas</roots>
      <film_styles>AR Rahman, Ilaiyaraaja, Anirudh, MM Keeravani, Vidyasagar</film_styles>
      <instruments>veena, nadaswaram, mridangam, ghatam, kanjira, gottuvadyam, thavil, flute</instruments>
      <folk>Kummi, Kolattam, Karakattam rhythms</folk>
      <scene_rules>
        <rule context="Devotional/temple">classical Carnatic with appropriate raga</rule>
        <rule context="Rural/village">folk percussion over orchestral</rule>
        <rule context="Urban/modern">fusion or electronic-acoustic blend</rule>
        <rule context="Action/sports">punchy mridangam + brass + high-energy rhythm</rule>
      </scene_rules>
    </region>

    <region name="NORTH INDIA (Hindi / Urdu / Bollywood)">
      <roots>Hindustani classical roots, raga-based compositions</roots>
      <film_styles>Bollywood styles: Golden Era strings -> modern hybrid orchestral</film_styles>
      <instruments>sitar, sarod, tabla, sarangi, bansuri, harmonium, shehnai</instruments>
      <folk>Bhojpuri dholak, Rajasthani khartal/morchang/algoza, Punjabi dhol/tumbi</folk>
    </region>

    <region name="BENGALI">
      <aesthetic>Rabindra Sangeet aesthetic, Baul mystic folk tradition</aesthetic>
      <instruments>dotara, khamak, ektara, esraj</instruments>
    </region>

    <region name="MARATHI">
      <styles>Dholki-driven Lavani rhythms, Powada ballad storytelling</styles>
      <devotional>Varkari devotional style for spiritual scenes</devotional>
    </region>

    <region name="SRI LANKAN">
      <styles>Kandyan drumming, raban folk percussion</styles>
      <instruments>geta beraya, rabana, horanewa</instruments>
    </region>

    <region name="PAKISTANI">
      <styles>Qawwali tradition (Nusrat influence), ghazal aesthetic</styles>
      <instruments>harmonium, tabla, dholak, sitar, rubab</instruments>
    </region>

    <region name="AFGHAN">
      <styles>Central Asian / Persian crossover, meditative drone</styles>
      <instruments>rubab, dutar, tabla, tula flute</instruments>
    </region>

    <region name="NEPALI / TIBETAN">
      <styles>Himalayan drone aesthetics, Buddhist ritual soundscapes</styles>
      <instruments>singing bowls, dungchen, damaru, Nepali sarangi</instruments>
    </region>

    <region name="SOUTHEAST ASIA">
      <country name="INDONESIAN">Gamelan (slendro/pelog tuning), angklung, gong textures</country>
      <country name="THAI">Pi phat ensemble, ranat xylophone, khong wong gongs</country>
      <country name="VIETNAMESE">Dan bau monochord, dan tranh zither, trong drums</country>
      <country name="FILIPINO">Kulintang gong ensemble, rondalla strings</country>
      <country name="CAMBODIAN">Pinpeat orchestra, chapei dong veng lute</country>
      <country name="BURMESE">Saung gauk arched harp, pat waing drum circle</country>
    </region>

    <region name="EAST ASIA">
      <country name="CHINESE">Pentatonic scales, erhu, guqin, pipa, guzheng, dizi, sheng, suona</country>
      <country name="JAPANESE">Gagaku, koto, shamisen, shakuhachi, taiko, biwa; Joe Hisaishi for modern</country>
      <country name="KOREAN">Pansori, gayageum, haegeum, daegeum, janggu, kkwaenggwari</country>
    </region>

    <region name="MIDDLE EAST AND NORTH AFRICA">
      <country name="ARABIC">Maqam (Hijaz/Rast/Bayati/Saba), oud, qanun, ney, darbuka, riq</country>
      <country name="PERSIAN">Dastgah system, tar, setar, santur, tombak, kamancheh</country>
      <country name="TURKISH">Ottoman makam, baglama, ney, kanun, davul, zurna</country>
      <country name="NORTH AFRICAN">Gnawa guembri/krakeb, Rai pop bendir/ghaita</country>
    </region>

    <region name="SUB-SAHARAN AFRICA">
      <subregion name="WEST">Kora, balafon, djembe, talking drum, ngoni; Afrobeats for urban</subregion>
      <subregion name="EAST">Tizita scale, krar, masenqo, kebero; Taarab for Swahili coast</subregion>
      <subregion name="SOUTHERN">Mbira, Isicathamiya choral, Marabi township jazz, Amapiano</subregion>
      <subregion name="CENTRAL">Likembe, ngoma drums, dense polyrhythmic layering</subregion>
    </region>

    <region name="LATIN AMERICA AND CARIBBEAN">
      <country name="BRAZILIAN">Samba/Bossa Nova/Baiao; cavaquinho, pandeiro, berimbau, sanfona</country>
      <country name="ARGENTINIAN">Tango bandoneon; chacarera/zamba folklore</country>
      <country name="ANDEAN">Pentatonic; siku, quena, charango, bombo</country>
      <country name="COLOMBIAN">Cumbia; vallenato accordion + caja + guacharaca</country>
      <country name="CUBAN">Son/mambo/rumba; trumpet, congas, tres, claves</country>
      <country name="JAMAICAN">Reggae/dancehall, nyahbinghi drumming</country>
      <country name="CARIBBEAN">Calypso, steel pan, soca</country>
    </region>

    <region name="EUROPE">
      <style name="CELTIC">Uilleann pipes, fiddle, bodhran, tin whistle, harp</style>
      <style name="FLAMENCO">Guitar, cajon, palmas, castanets; palo matches emotion</style>
      <style name="GREEK">Rebetiko bouzouki, Laika pop drama</style>
      <style name="BALKAN">Asymmetric meters (7/8, 9/8), gaida, tupan, cobza, cimbalom</style>
      <style name="SLAVIC">Balalaika, bayan accordion, gusli, Orthodox choral drone</style>
      <style name="NORDIC">Nyckelharpa, Hardanger fiddle, minimalist drone folk</style>
    </region>

    <region name="NORTH AMERICA">
      <style name="AMERICANA">Slide guitar/harmonica blues; banjo/dulcimer Appalachian; Gospel choir; New Orleans jazz brass</style>
      <style name="INDIGENOUS">Frame drum, hand drum, vocal chanting, flute</style>
    </region>
  </cultural_music_detection>

  <scene_based_overrides>
    <override scene="DEVOTIONAL / TEMPLE / RITUAL">modal, drone-heavy sacred music of that culture</override>
    <override scene="RURAL / VILLAGE / FOLK">acoustic folk instruments of that region</override>
    <override scene="URBAN / MODERN / STREET">fusion, electronic-acoustic, contemporary pop</override>
    <override scene="ACTION / BATTLE / SPORTS">punchy percussion + brass + driving rhythm</override>
    <override scene="ROMANCE / INTIMATE">solo melodic instrument + gentle harmonic bed</override>
    <override scene="GRIEF / FUNERAL / LOSS">slow, minor/modal, sparse</override>
    <override scene="CELEBRATION / FESTIVAL">layered percussion, call-and-response, bright tones</override>
    <override scene="MYSTERY / SUPERNATURAL">unusual scales, low drones, sparse texture</override>
    <override scene="EPIC / CINEMATIC">hybrid orchestral + culture's signature instrument as lead</override>
    <override scene="CHILDHOOD / INNOCENT">simple pentatonic or folk melody, light percussion</override>
  </scene_based_overrides>

  <emotion_arc_analysis>
    <instruction>Do NOT just pick one mood. Trace the emotional journey across the video timeline.</instruction>
    <instruction>Identify opening mood -> middle development -> closing resolution.</instruction>
  </emotion_arc_analysis>

  <output_format>
    <instruction>OUTPUT - Strict JSON, no extra text:</instruction>
    <schema>
{
  "video_context": {
    "language": "&lt;detected language&gt;",
    "content_type": "&lt;documentary | narrative | promotional | devotional | action | lifestyle | comedy | drama&gt;",
    "setting": "&lt;urban | rural | natural | indoor | temple | stadium | mixed&gt;",
    "narrative_tone": "&lt;inspirational | melancholic | celebratory | tense | peaceful | comedic | devotional&gt;"
  },
  "emotion_arc": {
    "opening": "&lt;emotion at start&gt;",
    "development": "&lt;emotion in middle&gt;",
    "resolution": "&lt;emotion at end&gt;"
  },
  "music_blueprint": {
    "duration_seconds": "&lt;float&gt;",
    "tempo_bpm": "&lt;e.g. 72-88 BPM, slow build to 110 BPM&gt;",
    "musical_scale": "&lt;raga name if Indian, maqam if Arabic, dastgah if Persian, or Western key&gt;",
    "genre": "&lt;specific genre e.g. Tamil folk fusion | Carnatic-electronic | Hindustani orchestral&gt;",
    "energy_curve": "&lt;e.g. low (3/10) -> medium (6/10) -> peak (8/10)&gt;",
    "primary_instruments": ["&lt;instrument 1&gt;", "&lt;instrument 2&gt;"],
    "secondary_instruments": ["&lt;instrument&gt;"],
    "rhythm_pattern": "&lt;e.g. Adi talam 8-beat | 4/4 western | Rupaka talam | 7/8 Balkan&gt;",
    "style_references": ["&lt;e.g. Ilaiyaraaja village bgm | AR Rahman emotional swell&gt;"],
    "cultural_authenticity_notes": "&lt;specific regional music conventions to follow&gt;",
    "avoid": "&lt;what NOT to include&gt;"
  },
  "scene_summary": "&lt;2-3 sentence plain English summary of what is happening in the video and why this music direction fits&gt;"
}
    </schema>
  </output_format>

  <strict_rules>
    <rule>Output ONLY valid JSON, no markdown, no explanation outside JSON</rule>
    <rule>Be specific - avoid generic answers like 'orchestral background music'</rule>
    <rule>cultural_authenticity_notes must reflect actual regional music traditions</rule>
  </strict_rules>
</system>"""

    def generate_prompt(self, analysis: Dict, celebrity_info: Dict = None) -> Dict:
        """LLM CALL #1: Deep video analysis -> Structured music blueprint (JSON)"""
        print("=" * 70)
        print("LLM CALL #1 - VIDEO ANALYSIS -> MUSIC BLUEPRINT (JSON)")
        print("=" * 70)

        user_message  = self._create_user_message(analysis, celebrity_info=celebrity_info)
        summary       = analysis["summary"]
        transcription = analysis["transcription"]

        print(f"  Video duration : {summary['duration_seconds']}s")
        print(f"  Language       : {transcription.get('language', 'unknown')}")
        print(f"  Sending to Bedrock...\n")

        try:
            raw_text  = self._invoke_model({
                "anthropic_version": "bedrock-2023-05-31",
                "max_tokens": 2000,
                "temperature": 0.5,
                "system": self.create_system_prompt(),
                "messages": [{"role": "user", "content": user_message}],
            })
            blueprint = self._extract_json(raw_text)

            mb = blueprint.get("music_blueprint", {})
            vc = blueprint.get("video_context", {})
            print("Music Blueprint Generated!")
            print(f"  Content Type  : {vc.get('content_type', 'N/A')}")
            print(f"  Language      : {vc.get('language', 'N/A')}")
            print(f"  Genre         : {mb.get('genre', 'N/A')}")
            print(f"  Tempo         : {mb.get('tempo_bpm', 'N/A')}")
            print(f"  Scale/Raga    : {mb.get('musical_scale', 'N/A')}")
            print(f"  Energy Curve  : {mb.get('energy_curve', 'N/A')}")
            print(f"  Instruments   : {', '.join(mb.get('primary_instruments', []))}")
            print(f"  Style Ref     : {', '.join(mb.get('style_references', []))}")
            print(f"  Scene Summary : {blueprint.get('scene_summary', 'N/A')}\n")
            return blueprint

        except Exception as e:
            print(f"Bedrock error in LLM Call #1: {e}\n-> Using fallback blueprint.\n")
            return self._fallback_prompt(analysis)

    def _create_user_message(self, analysis: Dict, celebrity_info: Dict = None) -> str:
        summary       = analysis["summary"]
        transcription = analysis["transcription"]
        frames        = analysis.get("frame_by_frame_analysis", [])

        if frames:
            indices       = [0, len(frames) // 2, len(frames) - 1]
            sample_frames = [frames[i] for i in indices if i < len(frames)]
        else:
            sample_frames = []

        msg = f"""Analyze the following video data and generate a culturally-aware music blueprint.

VIDEO METADATA
Duration         : {summary['duration_seconds']} seconds
Detected Language: {transcription.get('language', 'unknown')}
Has Dialogue     : {summary['has_audio']}

TRANSCRIPTION (first 600 chars)
{transcription.get('text', 'No audio detected')[:600]}

VISUAL CHARACTERISTICS
Pacing           : {summary['motion_analysis']['pacing']}
Motion Intensity : {summary['motion_analysis']['average_intensity']}
Color Temperature: {summary['color_analysis']['dominant_temperature']}

EMOTIONS DETECTED (top 5)
{json.dumps(summary['detected_emotions'][:5], indent=2)}

KEY OBJECTS IN SCENE (top 10)
{json.dumps(summary['common_objects'][:10], indent=2)}

DOMINANT SCENE TYPES (top 3)
{json.dumps(summary['dominant_scenes'][:3], indent=2)}

FRAME-BY-FRAME SAMPLES (start / mid / end)"""

        for label, frame in zip(["START", "MIDDLE", "END"], sample_frames):
            msg += f"""
[{label} - {frame['timestamp']}s]
  Caption : {frame['caption']}
  Objects : {', '.join([obj['object'] for obj in frame.get('objects', [])[:6]])}
"""
        # Add celebrity & preferred music director info if available
        if celebrity_info and celebrity_info.get("preferred_music_directors"):
            msg += "\n\nPREFERRED MUSIC DIRECTORS (from celebrity analysis):\n"
            msg += "These music directors were identified as the most frequent hit-givers "
            msg += "for celebrities detected in this video. USE THEIR STYLE AS PRIMARY REFERENCE.\n"
            for md in celebrity_info["preferred_music_directors"][:3]:
                msg += f"  - {md['name']}: {md.get('style_description', 'N/A')}\n"
                msg += f"    Signature instruments: {', '.join(md.get('signature_instruments', []))}\n"
                msg += f"    Notable works: {', '.join(md.get('notable_works', [])[:3])}\n"
            if celebrity_info.get("detected_celebrities"):
                msg += "\nDETECTED CELEBRITIES:\n"
                for c in celebrity_info["detected_celebrities"][:3]:
                    msg += f"  - {c['name']} ({c.get('role', 'unknown')})\n"
            msg += "\nIMPORTANT: The music blueprint MUST use the preferred music directors above "
            msg += "as primary style_references. Their musical style should heavily influence "
            msg += "the genre, instruments, and overall feel of the BGM.\n"

        msg += "\nUsing all the above, generate the MUSIC BLUEPRINT JSON."
        return msg

    def _extract_json(self, text: str) -> Dict:
        start = text.find("{")
        end   = text.rfind("}") + 1
        if start != -1 and end > start:
            return json.loads(text[start:end])
        raise ValueError("No valid JSON block found in LLM response")

    def _fallback_prompt(self, analysis: Dict) -> Dict:
        s        = analysis["summary"]
        lang     = analysis["transcription"].get("language", "unknown")
        is_tamil = "tamil" in lang.lower()
        return {
            "video_context": {
                "language": lang,
                "content_type": "general",
                "setting": "mixed",
                "narrative_tone": "balanced"
            },
            "emotion_arc": {"opening": "calm", "development": "engaging", "resolution": "uplifting"},
            "music_blueprint": {
                "duration_seconds": s["duration_seconds"],
                "tempo_bpm": "90-110 BPM",
                "musical_scale": "Kapi raga" if is_tamil else "C major",
                "genre": "Tamil cinematic folk fusion" if is_tamil else "Cinematic background",
                "energy_curve": "medium (5/10) -> medium-high (7/10)",
                "primary_instruments": ["veena", "flute", "mridangam"] if is_tamil else ["strings", "flute", "light percussion"],
                "secondary_instruments": ["strings", "ghatam"] if is_tamil else ["piano", "pads"],
                "rhythm_pattern": "Adi talam 8-beat" if is_tamil else "4/4 moderate",
                "style_references": ["Ilaiyaraaja village bgm style"] if is_tamil else ["Generic cinematic"],
                "cultural_authenticity_notes": "Incorporate South Indian Carnatic melodic sensibility with folk undertones" if is_tamil else "Universal",
                "avoid": "heavy western drums, EDM drops, distortion guitar"
            },
            "scene_summary": "General video content with balanced pacing. Music should support the narrative without overpowering dialogue."
        }

    # -------------------------------------------------------------------------
    # LLM CALL #2 - Music Blueprint (JSON) -> MusicGen Prompt (Text, max 500 chars)
    # -------------------------------------------------------------------------

    def convert_to_music_gen_prompt(self, bgm_prompt: Dict) -> str:
        """LLM CALL #2: Music blueprint (JSON) -> MusicGen-optimized text prompt (max 500 chars)"""
        print("=" * 70)
        print("LLM CALL #2 - MUSIC BLUEPRINT -> MUSICGEN PROMPT (TEXT)")
        print("=" * 70)

        system_prompt = """<system>
  <role>You are a world-class specialist in writing text prompts for MusicGen (Meta's text-to-music AI model).</role>

  <task>You receive a structured MUSIC BLUEPRINT (JSON) and convert it into one perfectly optimized MusicGen prompt.</task>

  <prompt_construction_rules>

    <structure>
      <instruction>Follow this order, comma-separated</instruction>

      <step number="1" name="CULTURAL ORIGIN + TRADITION">
        <instruction>Name the specific musical tradition, not just the country.</instruction>
        <examples>South Indian Carnatic, West African griot tradition, Ottoman classical, Andean indigenous folk, Celtic Irish trad, Afrobeats Lagos urban, Flamenco Andalusian, Hindustani classical</examples>
      </step>

      <step number="2" name="SPECIFIC GENRE / SUBGENRE">
        <instruction>Be precise - name the exact style within the tradition.</instruction>
        <examples>Tamil folk fusion, Bossa Nova, Flamenco Solea, Qawwali devotional, Gamelan Gong Kebyar, Rebetiko, Amapiano, Son Cubano, Cumbia, Baiao, Bhairavi thumri</examples>
      </step>

      <step number="3" name="MODAL / SCALE / RAGA / MAQAM / DASTGAH">
        <instruction>Always name the exact scale system used.</instruction>
        <system name="Indian">Raga Bhairavi, Raga Kapi, Raga Desh, Raga Yaman</system>
        <system name="Arabic">Maqam Hijaz, Maqam Rast, Maqam Bayati, Maqam Saba</system>
        <system name="Persian">Dastgah Shur, Dastgah Chahargah, Dastgah Mahur</system>
        <system name="Western">D Dorian, F minor, C Lydian, B Phrygian</system>
        <system name="African">Tizita scale, Ethiopian pentatonic</system>
        <system name="East Asian">Chinese pentatonic, Japanese In scale, Korean Gyemyeonjo</system>
      </step>

      <step number="4" name="RHYTHMIC FEEL / TALA / METER / GROOVE">
        <instruction>Name the exact rhythmic system.</instruction>
        <system name="Indian">Adi talam 8-beat, Rupaka talam 6-beat, Khanda chapu</system>
        <system name="Arabic">Maqsum 8-beat, Wahda 4-beat, Masmoudi 8-beat</system>
        <system name="Western">4/4 steady groove, 3/4 waltz, 6/8 compound</system>
        <system name="Balkan">7/8 asymmetric, 9/8 Bulgarian, 11/16 Macedonian</system>
        <system name="African">12/8 compound polyrhythm, clave 3-2 pattern</system>
        <system name="Latin">Son clave 3-2, Bossa Nova 2-3 clave, Cumbia 4/4 groove</system>
      </step>

      <step number="5" name="EMOTION ARC (opening -> development -> resolution)">
        <instruction>Describe as a cinematic journey, not a single word.</instruction>
        <examples>
          <example>hushed reverence opening -> swelling devotion -> transcendent release</example>
          <example>playful curiosity -> rising tension -> triumphant breakthrough</example>
          <example>quiet grief -> searching longing -> gentle acceptance</example>
          <example>meditative stillness -> gathering momentum -> joyful resolution</example>
        </examples>
      </step>

      <step number="6" name="TEMPO DESCRIPTOR">
        <instruction>Always include BPM + feel descriptor.</instruction>
        <examples>slow 65 BPM rubato, steady 90 BPM flowing, driving 120 BPM relentless, free tempo opening then 88 BPM settled</examples>
      </step>

      <step number="7" name="PRIMARY INSTRUMENTS (name each specifically)">
        <rules>
          <rule>NEVER say "strings" alone -> say "South Indian bamboo flute", "erhu", "kora"</rule>
          <rule>NEVER say "drums" alone -> say "mridangam", "djembe", "darbuka", "taiko"</rule>
          <rule>NEVER say "synth" alone -> say "analog pad", "granular texture", "sub-bass drone"</rule>
        </rules>
        <by_tradition>
          <tradition name="Indian">veena, mridangam, ghatam, kanjira, nadaswaram, South Indian bamboo flute</tradition>
          <tradition name="Arabic">oud, qanun, ney, darbuka, riq</tradition>
          <tradition name="African">kora, djembe, talking drum, mbira, balafon, krar</tradition>
          <tradition name="East Asian">erhu, guzheng, koto, shamisen, gayageum, pipa</tradition>
          <tradition name="Celtic">uilleann pipes, fiddle, bodhran, tin whistle, harp</tradition>
          <tradition name="Latin">nylon string guitar, cajon, bongo, congas, marimba, cuatro</tradition>
          <tradition name="Balkan">gaida, tupan, cobza, cimbalom, zurna</tradition>
          <tradition name="Electronic">Moog analog synth, granular pad, ambient drone, 808 sub-bass</tradition>
        </by_tradition>
      </step>

      <step number="8" name="SECONDARY / TEXTURAL INSTRUMENTS">
        <instruction>Supporting layers, atmosphere, pads, counter-melody instruments.</instruction>
      </step>

      <step number="9" name="TEXTURE + ATMOSPHERE">
        <instruction>Pick 2-3 from this list that match the scene.</instruction>
        <options>sparse, intimate, meditative, brooding, luminous, vast, warm, cold, hypnotic, layered, dense, driving, pulsating, floating, ethereal, grounded, ceremonial, joyful, mournful, urgent, triumphant, mysterious, devotional, playful, raw, cinematic, contemplative</options>
      </step>

      <step number="10" name="PRODUCTION / MIX STYLE">
        <instruction>Pick the one that fits.</instruction>
        <options>
          <option>organic acoustic recording</option>
          <option>field recording ambience</option>
          <option>cinematic hybrid orchestral</option>
          <option>modern Bollywood production</option>
          <option>lo-fi warm analog</option>
          <option>clean studio recording</option>
          <option>live ensemble feel</option>
          <option>traditional court music aesthetic</option>
          <option>Carnatic concert hall reverb</option>
          <option>world music fusion production</option>
          <option>contemporary film score mix</option>
          <option>intimate close-mic recording</option>
          <option>large hall cathedral reverb</option>
        </options>
      </step>

      <step number="11" name="VOCAL DIRECTIVE (always include one of these)">
        <options>
          <option>no vocals, instrumental only</option>
          <option>wordless vocal hum only, no lyrics</option>
          <option>choral drone, no lyrics</option>
          <option>call-and-response vocal texture, no lyrics</option>
        </options>
      </step>
    </structure>

    <scene_type_overrides>
      <instruction>Apply these priorities on top of the cultural base.</instruction>
      <override scene="ACTION / SPORTS">percussion-first, punchy brass stabs, relentless forward momentum</override>
      <override scene="DEVOTIONAL / RITUAL">drone root note anchored, modal melody, reverberant sacred space</override>
      <override scene="ROMANCE">solo melodic lead, gentle harmonic swell, intimate mic placement</override>
      <override scene="GRIEF / LOSS">very slow tempo, sparse texture, minor or modal, long resonant notes</override>
      <override scene="CELEBRATION">bright tonal palette, layered percussion, call-and-response motifs</override>
      <override scene="MYSTERY">low drones, extended techniques, unusual scales, wide stereo space</override>
      <override scene="CHILDHOOD / INNOCENT">simple pentatonic melody, light percussion, playful timbre</override>
      <override scene="EPIC / CINEMATIC">full ensemble swell, culture's signature instrument as melodic hero</override>
      <override scene="RURAL / FOLK">acoustic only, no electronics, authentic folk recording feel</override>
      <override scene="URBAN / MODERN">fusion elements, light electronic texture blended with acoustic</override>
    </scene_type_overrides>

    <instrument_specificity_rules>
      <rule content="Indian content">name exact raga + tala + instrument (not "flute" -> "South Indian bamboo flute")</rule>
      <rule content="Arabic content">name the maqam + specific percussion (not "drum" -> "darbuka" or "riq")</rule>
      <rule content="African content">name exact instrument (not "drums" -> "djembe", "talking drum", "mbira")</rule>
      <rule content="East Asian">name exact instrument (not "string" -> "erhu", "guzheng", "koto")</rule>
      <rule content="Celtic">name exact instrument (not "wind" -> "uilleann pipes", "tin whistle")</rule>
      <rule content="Latin">name rhythm style AND instrument ("Cuban tres guitar, 3-2 clave rhythm")</rule>
      <rule content="Electronic">name synthesis type ("analog Moog pad", "granular texture", "sub-bass drone")</rule>
    </instrument_specificity_rules>

    <strict_avoidance>
      <never>Write: "beautiful music", "nice melody", "background music", "soothing sounds", "pleasant", "relaxing music", "good vibes" - these are useless to MusicGen</never>
      <never>Use vague instrument names when specific ones are available</never>
      <never>Mix culturally contradictory elements unless it is intentional fusion</never>
      <never>Output more than 500 characters</never>
      <never>Output anything except the single prompt line - no quotes, no markdown, no explanation</never>
    </strict_avoidance>

    <output_format>
      <instruction>One single line of comma-separated descriptors.</instruction>
      <instruction>No quotes. No markdown. No explanation. No preamble. No numbering.</instruction>
      <instruction>Maximum 500 characters - use every character to be maximally specific and descriptive.</instruction>
    </output_format>

  </prompt_construction_rules>
</system>"""

        mb  = bgm_prompt.get("music_blueprint", {})
        vc  = bgm_prompt.get("video_context", {})
        arc = bgm_prompt.get("emotion_arc", {})

        focused_input = {
            "content_type":                vc.get("content_type"),
            "language_culture":            vc.get("language"),
            "narrative_tone":              vc.get("narrative_tone"),
            "setting":                     vc.get("setting"),
            "emotion_arc":                 arc,
            "genre":                       mb.get("genre"),
            "tempo_bpm":                   mb.get("tempo_bpm"),
            "musical_scale_or_raga":       mb.get("musical_scale"),
            "energy_curve":                mb.get("energy_curve"),
            "primary_instruments":         mb.get("primary_instruments"),
            "secondary_instruments":       mb.get("secondary_instruments"),
            "rhythm_pattern":              mb.get("rhythm_pattern"),
            "style_references":            mb.get("style_references"),
            "cultural_authenticity_notes": mb.get("cultural_authenticity_notes"),
            "avoid":                       mb.get("avoid"),
            "scene_summary":               bgm_prompt.get("scene_summary"),
        }

        user_message = f"""Convert this music blueprint into a single MusicGen prompt:

{json.dumps(focused_input, indent=2)}

Follow the construction rules exactly.
Output one line only. Max 500 characters.
Be maximally specific: cultural tradition, exact raga/maqam/scale, tala/rhythm, named instruments, emotion arc as a journey, tempo, texture, production style, vocal directive.
No explanations. No preamble. Just the prompt."""

        try:
            print("  Converting blueprint to MusicGen prompt...")
            raw_text = self._invoke_model({
                "anthropic_version": "bedrock-2023-05-31",
                "max_tokens": 600,
                "temperature": 0.4,
                "system": system_prompt,
                "messages": [{"role": "user", "content": user_message}],
            })

            # Sanitize output
            music_gen_prompt = (
                raw_text
                .replace("**", "").replace("*", "")
                .replace('"', "").replace("'", "")
                .split("\n")[0]
                .strip()
            )

            # Enforce 500 character limit
            if len(music_gen_prompt) > 500:
                music_gen_prompt = music_gen_prompt[:497] + "..."

            print(f"MusicGen Prompt Ready!")
            print(f"  Characters : {len(music_gen_prompt)} / 500")
            print(f"  -> {music_gen_prompt}\n")
            return music_gen_prompt

        except Exception as e:
            print(f"LLM Call #2 failed: {e}\n-> Using rule-based fallback.\n")
            return self._fallback_music_gen_prompt(bgm_prompt)

    def _fallback_music_gen_prompt(self, bgm_prompt: Dict) -> str:
        """Fallback rule-based conversion if LLM Call #2 fails"""
        mb  = bgm_prompt.get("music_blueprint", {})
        arc = bgm_prompt.get("emotion_arc", {})
        parts = [
            mb.get("genre", "cinematic background"),
            mb.get("musical_scale", ""),
            mb.get("rhythm_pattern", ""),
            f"{arc.get('opening', 'calm')} to {arc.get('resolution', 'uplifting')} mood",
            mb.get("tempo_bpm", "90-110 BPM"),
            ", ".join(mb.get("primary_instruments", ["strings", "flute"])),
            ", ".join(mb.get("secondary_instruments", [])),
            mb.get("cultural_authenticity_notes", ""),
            "instrumental only, no vocals"
        ]
        return ", ".join(filter(None, parts))[:500]


print("BedrockBGMPromptGenerator (expanded MusicGen rules + API Key auth) defined.")

BedrockBGMPromptGenerator (expanded MusicGen rules + API Key auth) defined.


## 5. SimpleTextToMusicGenerator

In [13]:
class SimpleTextToMusicGenerator:
    """Generate music from text using MusicGen via transformers."""

    def __init__(self, model_name: str = "facebook/musicgen-small"):
        print(f"Loading model: {model_name}")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {self.device}")
        if self.device == "cpu":
            print("⚠ Running on CPU - generation will be SLOW")
        print("Loading processor...")
        self.processor = AutoProcessor.from_pretrained(model_name)
        print("Loading model...")
        self.model = MusicgenForConditionalGeneration.from_pretrained(
            model_name, torch_dtype=torch.float16 if self.device == "cuda" else torch.float32
        ).to(self.device)
        self.model.eval()
        self.model_name = model_name
        self.sampling_rate = self.model.config.audio_encoder.sampling_rate
        print(f"✓ {model_name} loaded! Sampling rate: {self.sampling_rate} Hz\n")

    def generate(self, prompt: str, duration: float = 30.0, guidance_scale: float = 3.0, max_new_tokens=None) -> torch.Tensor:
        print(f"Generating music...\nPrompt: {prompt}\nDuration: {duration}s")
        inputs = self.processor(text=[prompt], padding=True, return_tensors="pt").to(self.device)
        if max_new_tokens is None:
            max_new_tokens = int(duration * 50)
        print(f"Generating {max_new_tokens} tokens...")
        with torch.no_grad():
            audio_values = self.model.generate(**inputs, max_new_tokens=max_new_tokens, guidance_scale=guidance_scale, do_sample=True, temperature=1.0)
        print("✓ Generation complete!\n")
        return audio_values

    def save_audio(self, audio: torch.Tensor, output_path: str):
        audio_np = audio.cpu().numpy().squeeze()
        audio_np = (audio_np * 32767).astype("int16")
        scipy.io.wavfile.write(output_path, self.sampling_rate, audio_np)
        print(f"✓ Audio saved to: {output_path} ({len(audio_np) / self.sampling_rate:.2f}s)")

print("SimpleTextToMusicGenerator defined.")


SimpleTextToMusicGenerator defined.


## 6. VideoMusicPipeline

In [18]:
class VideoMusicPipeline:
    """Complete pipeline: Video Analysis → BGM Generation → Audio Mixing → Final Video."""

    def __init__(self, whisper_model="base", music_model="facebook/musicgen-small", use_bedrock=True):
        print("=" * 70)
        print("COMPLETE VIDEO MUSIC PIPELINE")
        print("=" * 70 + "\n")
        print("Step 1: Initializing Video Analyzer...")
        self.video_analyzer = CompleteVideoAnalyzer(whisper_model=whisper_model)
        self.use_bedrock = use_bedrock
        if use_bedrock:
            print("\nStep 2: Initializing AWS Bedrock BGM Prompt Generator...")
            try:
                self.bgm_generator = BedrockBGMPromptGenerator()
            except Exception as e:
                print(f"⚠ Bedrock init failed: {e}\nFalling back to basic prompt generation")
                self.use_bedrock = False
        # Celebrity & Music Director Analyzer
        self.celebrity_analyzer = None
        if use_bedrock:
            print("\nStep 2.5: Initializing Celebrity & Music Director Analyzer...")
            try:
                self.celebrity_analyzer = CelebrityMusicDirectorAnalyzer()
            except Exception as e:
                print(f"⚠ Celebrity analyzer init failed: {e}")
        print("\nStep 3: Initializing Music Generator...")
        self.music_generator = SimpleTextToMusicGenerator(model_name=music_model)
        print("\nPIPELINE READY!\n")

    def generate_bgm_prompt_from_analysis(self, analysis: Dict) -> Dict:
        """Basic BGM prompt generation (no Bedrock)."""
        summary = analysis["summary"]
        duration = summary["duration_seconds"]
        has_audio = summary["has_audio"]
        pacing = summary["motion_analysis"]["pacing"]
        dominant_emotions = [e["emotion"] for e in summary["detected_emotions"][:3]]
        color_temp = summary["color_analysis"]["dominant_temperature"]
        emotion_to_mood = {"happy": "cheerful and upbeat", "sad": "melancholic and gentle", "surprise": "energetic and dynamic", "fear": "tense and dramatic", "neutral": "calm and balanced", "angry": "intense and powerful"}
        mood_descriptors = [emotion_to_mood.get(e, "balanced") for e in dominant_emotions[:2]]
        overall_mood = ", ".join(mood_descriptors) if mood_descriptors else "balanced"
        tempo_map = {"slow-paced": "slow tempo (60-80 BPM)", "moderate-paced": "moderate tempo (90-110 BPM)", "fast-paced": "fast tempo (120-140 BPM)"}
        tempo = tempo_map.get(pacing, "moderate tempo")
        if color_temp == "warm":
            instruments = "warm acoustic instruments, soft guitar, gentle piano"
        elif color_temp == "cool":
            instruments = "ambient synths, soft pads, subtle electronic elements"
        else:
            instruments = "balanced mix of acoustic and electronic, light percussion"
        if has_audio:
            prompt = f"Subtle background music, {overall_mood}, {tempo}, {instruments}, soft and unobtrusive, suitable for voice-over, gentle dynamics, ambient style, {int(duration)} seconds"
        else:
            prompt = f"Background music, {overall_mood}, {tempo}, {instruments}, cinematic, {int(duration)} seconds"
        return {"duration_seconds": duration, "has_dialogue": has_audio, "pacing": pacing, "dominant_emotions": dominant_emotions, "color_temperature": color_temp, "generated_prompt": prompt, "music_direction": {"mood": overall_mood, "tempo": tempo, "instrumentation": instruments, "style": "subtle background" if has_audio else "cinematic"}}

    def check_video_has_audio(self, video_path: str) -> bool:
        try:
            result = subprocess.run(["ffmpeg", "-i", video_path, "-hide_banner"], capture_output=True, text=True)
            return "Audio:" in result.stderr
        except:
            return False

    def mix_audio(self, original_video_path, bgm_path, output_path, bgm_volume=0.15, voice_volume=1.0):
        print("AUDIO PROCESSING")
        has_audio = self.check_video_has_audio(original_video_path)
        if has_audio:
            print(f"✓ Video has audio - mixing (voice={voice_volume}, bgm={bgm_volume})")
            cmd = ["ffmpeg", "-i", original_video_path, "-i", bgm_path, "-filter_complex",
                   f"[0:a]volume={voice_volume}[a0];[1:a]volume={bgm_volume}[a1];[a0][a1]amix=inputs=2:duration=first:dropout_transition=2",
                   "-ac", "2", "-y", output_path]
        else:
            print("✓ Video has NO audio - using BGM as primary audio")
            cmd = ["ffmpeg", "-i", bgm_path, "-ac", "2", "-y", output_path]
        try:
            subprocess.run(cmd, capture_output=True, text=True, check=True)
            print(f"✓ Output: {output_path}\n")
            return True
        except subprocess.CalledProcessError as e:
            print(f"❌ Audio processing failed: {e.stderr}")
            return False

    def create_final_video(self, original_video_path, mixed_audio_path, output_video_path):
        print("CREATING FINAL VIDEO")
        cmd = ["ffmpeg", "-i", original_video_path, "-i", mixed_audio_path, "-c:v", "copy", "-c:a", "aac", "-b:a", "192k", "-map", "0:v:0", "-map", "1:a:0", "-shortest", "-y", output_video_path]
        try:
            subprocess.run(cmd, capture_output=True, text=True, check=True)
            print(f"✓ Final video: {output_video_path}\n")
            return True
        except subprocess.CalledProcessError as e:
            print(f"❌ Video creation failed: {e.stderr}")
            return False

    def process_video(self, video_path, output_dir=None, bgm_volume=0.15, fps=1, skip_music_generation=False, bgm_path=None):
        print("\nSTARTING COMPLETE VIDEO MUSIC PIPELINE\n")
        video_path = Path(video_path)
        if output_dir is None:
            output_dir = video_path.parent
        else:
            output_dir = Path(output_dir)
            output_dir.mkdir(exist_ok=True)
        vn = video_path.stem
        analysis_json = output_dir / f"{vn}_analysis.json"
        bgm_prompt_json = output_dir / f"{vn}_bgm_prompt.json"
        bgm_audio = output_dir / f"{vn}_bgm.wav"
        mixed_audio = output_dir / f"{vn}_mixed_audio.wav"
        final_video = output_dir / f"{vn}_with_bgm.mp4"
        results = {"input_video": str(video_path), "output_directory": str(output_dir)}

        # Step 1: Analyze
        print("STEP 1: ANALYZING VIDEO")
        analysis = self.video_analyzer.analyze_video(str(video_path), fps=fps)
        analysis = convert_to_native(analysis)
        with open(analysis_json, "w", encoding="utf-8") as f:
            json.dump(analysis, f, indent=2, ensure_ascii=False)
        print(f"✓ Analysis saved: {analysis_json}\n")
        results["analysis"] = str(analysis_json)

        # Step 1.5: Celebrity & Music Director Analysis
        celebrity_info = None
        if self.celebrity_analyzer:
            print("STEP 1.5: CELEBRITY & MUSIC DIRECTOR ANALYSIS")
            try:
                celebrity_info = self.celebrity_analyzer.analyze_celebrities(analysis)
                analysis["celebrity_analysis"] = celebrity_info
                # Re-save analysis with celebrity info
                with open(analysis_json, "w", encoding="utf-8") as f:
                    json.dump(analysis, f, indent=2, ensure_ascii=False)
                print(f"✓ Celebrity analysis added to: {analysis_json}\n")
                results["celebrity_analysis"] = celebrity_info
            except Exception as e:
                print(f"⚠ Celebrity analysis failed: {e}\n")

        # Step 2: BGM prompt
        print("STEP 2: GENERATING BGM PROMPT")
        if self.use_bedrock:
            bgm_prompt = self.bgm_generator.generate_prompt(analysis, celebrity_info=celebrity_info)
        else:
            bgm_prompt = self.generate_bgm_prompt_from_analysis(analysis)
        with open(bgm_prompt_json, "w", encoding="utf-8") as f:
            json.dump(bgm_prompt, f, indent=2, ensure_ascii=False)
        print(f"✓ BGM prompt saved: {bgm_prompt_json}\n")
        results["bgm_prompt"] = str(bgm_prompt_json)

        # Step 3: Generate music
        if skip_music_generation and bgm_path:
            print("STEP 3: USING EXISTING BGM")
            bgm_audio = Path(bgm_path)
        else:
            print("STEP 3: GENERATING MUSIC")
            if self.use_bedrock and isinstance(bgm_prompt, dict) and "genre" in bgm_prompt:
                prompt = self.bgm_generator.convert_to_music_gen_prompt(bgm_prompt)
            elif "generated_prompt" in bgm_prompt:
                prompt = bgm_prompt["generated_prompt"]
            else:
                prompt = str(bgm_prompt)
            duration = bgm_prompt.get("duration", bgm_prompt.get("duration_seconds", 30))
            audio = self.music_generator.generate(prompt, duration=duration)
            self.music_generator.save_audio(audio, str(bgm_audio))
        results["bgm_audio"] = str(bgm_audio)

        # Step 4: Mix audio
        print("STEP 4: MIXING AUDIO")
        success = self.mix_audio(str(video_path), str(bgm_audio), str(mixed_audio), bgm_volume=bgm_volume)
        if not success:
            print("⚠ Audio mixing failed")
            return results
        results["mixed_audio"] = str(mixed_audio)

        # Step 5: Final video
        print("STEP 5: CREATING FINAL VIDEO")
        success = self.create_final_video(str(video_path), str(mixed_audio), str(final_video))
        if success:
            results["final_video"] = str(final_video)

        print("\nPIPELINE COMPLETE!")
        for k, v in results.items():
            print(f"  {k}: {v}")
        return results

print("VideoMusicPipeline defined.")


VideoMusicPipeline defined.


## 7. Run the Full Pipeline

Option A runs everything end-to-end. Option B below lets you run step-by-step.

### Option A: One-Shot (Full Pipeline)

In [ ]:
pipeline = VideoMusicPipeline(
    whisper_model=WHISPER_MODEL,
    music_model=MUSIC_MODEL,
    use_bedrock=USE_BEDROCK
)

results = pipeline.process_video(
    VIDEO_PATH,
    output_dir=OUTPUT_DIR,
    bgm_volume=BGM_VOLUME,
    fps=FPS
)

print("\nResults:")
for key, value in results.items():
    print(f"  {key}: {value}")

COMPLETE VIDEO MUSIC PIPELINE

Step 1: Initializing Video Analyzer...
INITIALIZING COMPLETE VIDEO ANALYZER
Using device: cuda

Loading Whisper model...


100%|███████████████████████████████████████| 139M/139M [00:02<00:00, 69.9MiB/s]


✓ Whisper loaded

Loading CLIP model...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

MusicgenForConditionalGeneration LOAD REPORT from: facebook/musicgen-small
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decoder.model.decoder.embed_positions.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

✓ facebook/musicgen-small loaded! Sampling rate: 32000 Hz


PIPELINE READY!


STARTING COMPLETE VIDEO MUSIC PIPELINE

STEP 1: ANALYZING VIDEO

COMPLETE VIDEO ANALYSIS PIPELINE

STEP 1: AUDIO TRANSCRIPTION
⚠ Audio transcription failed: Failed to load audio: ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Analyzing frame 7/20 at 6.00s...
Analyzing frame 8/20 at 7.00s...
Analyzing frame 9/20 at 8.00s...
Analyzing frame 10/20 at 9.00s...
Analyzing frame 11/20 at 10.00s...
Analyzing frame 12/20 at 11.00s...
Analyzing frame 13/20 at 12.00s...
Analyzing frame 14/20 at 13.00s...
Analyzing frame 15/20 at 14.00s...
Analyzing frame 16/20 at 15.00s...
Analyzing frame 17/20 at 16.00s...
Analyzing frame 18/20 at 17.00s...
Analyzing frame 19/20 at 18.00s...
Analyzing frame 20/20 at 19.00s...
✓ Frame analysis complete

STEP 4: GENERATING SUMMARY
✓ Summary generated

✓ Analysis saved: /kaggle/working/output/12142366-uhd_4096_2160_24fps_with_bgm (online-video-cutter.com) (1)_analysis.json

STEP 1.5: CELEBRITY & MUSIC DIRECTOR ANALYSIS
CELEBRITY & MUSIC DIRECTOR ANALYSIS
  Sending video context to Bedrock for celebrity identification...
  ⚠ Celebrity analysis failed: Bedrock API error 403: {"Message":"Authentication failed: Please make sure your API Key is valid."}
  Falling back to language-based music

---
### Option B: Step-by-Step

#### Step 1: Video Analysis

In [10]:
analyzer = CompleteVideoAnalyzer(whisper_model=WHISPER_MODEL)
analysis = analyzer.analyze_video(VIDEO_PATH, fps=FPS)
analysis = convert_to_native(analysis)

analysis_path = f"{OUTPUT_DIR}/{video_name}_analysis.json"
with open(analysis_path, "w", encoding="utf-8") as f:
    json.dump(analysis, f, indent=2, ensure_ascii=False)
print(f"Saved: {analysis_path}")

INITIALIZING COMPLETE VIDEO ANALYZER
Using device: cuda

Loading Whisper model...
✓ Whisper loaded

Loading CLIP model...
Failed with openai/clip-vit-base-patch32: CLIPModel.__init__() got an unexpected keyword argument 'resume_download'
Trying alternative: openai/clip-vit-base-patch16...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ CLIP loaded

Loading BLIP captioning model...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

...

DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Object detector loaded

ALL MODELS LOADED!


COMPLETE VIDEO ANALYSIS PIPELINE

STEP 1: AUDIO TRANSCRIPTION
Detected language: English


100%|██████████| 1595/1595 [00:00<00:00, 29317.43frames/s]

✓ Detected language: en
STEP 2: FRAME EXTRACTION
Video: 16.0s, 30.0 FPS, extracting at 1 FPS


✓ Extracted 16 frames

STEP 3: FRAME-BY-FRAME ANALYSIS
Analyzing frame 1/16 at 0.00s...
Analyzing frame 2/16 at 1.00s...
Analyzing frame 3/16 at 2.00s...
Analyzing frame 4/16 at 3.00s...
Analyzing frame 5/16 at 4.00s...
Analyzing frame 6/16 at 5.00s...
Analyzing frame 7/16 at 6.00s...
Analyzing frame 8/16 at 7.00s...
Analyzing frame 9/16 at 8.00s...
Analyzing frame 10/16 at 9.00s...
Analyzing frame 11/16 at 10.00s...
Analyzing frame 12/16 at 11.00s...
Analyzing frame 13/16 at 12.00s...
Analyzing frame 14/16 at 13.00s...
Analyzing frame 15/16 at 14.00s...
Analyzing frame 16/16 at 15.00s...
✓ Frame analysis complete

STEP 4: GENERATING SUMMARY
✓ Summary generated

Saved: /kaggle/working/output/Tamil culture wedding in srilanka _analysis.json


In [11]:
# Inspect analysis summary
summary = analysis["summary"]
print(f"Duration: {summary['duration_seconds']}s")
print(f"Frames: {summary['total_frames_analyzed']}")
print(f"Has audio: {summary['has_audio']}")
print(f"Pacing: {summary['motion_analysis']['pacing']}")
print(f"Color temp: {summary['color_analysis']['dominant_temperature']}")
print(f"Objects: {[o['object'] for o in summary['common_objects'][:5]]}")
print(f"Emotions: {[e['emotion'] for e in summary['detected_emotions'][:3]]}")
if analysis["transcription"].get("text"):
    print(f"\nTranscription: {analysis['transcription']['text'][:300]}")

Duration: 15.0s
Frames: 16
Has audio: True
Pacing: fast-paced
Color temp: warm
Objects: ['person', 'donut', 'bottle', 'cake', 'tie']
Emotions: ['sad', 'fear', 'happy']

Transcription: Music


#### Step 2: BGM Prompt Generation

In [12]:
if USE_BEDROCK:
    bgm_generator = BedrockBGMPromptGenerator()
    bgm_prompt = bgm_generator.generate_prompt(analysis)
else:
    # Basic prompt generation (no AWS needed)
    _p = VideoMusicPipeline.__new__(VideoMusicPipeline)
    bgm_prompt = _p.generate_bgm_prompt_from_analysis(analysis)

prompt_path = f"{OUTPUT_DIR}/{video_name}_bgm_prompt.json"
with open(prompt_path, "w", encoding="utf-8") as f:
    json.dump(bgm_prompt, f, indent=2, ensure_ascii=False)
print(f"Saved: {prompt_path}")
print(json.dumps(bgm_prompt, indent=2))

Initializing Bedrock client with API Key...
Connected to Endpoint : https://bedrock-runtime.us-east-1.amazonaws.com/model/arn%3Aaws%3Abedrock%3Aus-east-1%3A926251048803%3Aapplication-inference-profile%2Faw4utpbetipp/invoke
Model                 : arn:aws:bedrock:us-east-1:926251048803:application-inference-profile/aw4utpbetipp

LLM CALL #1 - VIDEO ANALYSIS -> MUSIC BLUEPRINT (JSON)
  Video duration : 15.0s
  Language       : en
  Sending to Bedrock...

Music Blueprint Generated!
  Content Type  : devotional
  Language      : English
  Genre         : South Indian Carnatic devotional with folk percussion layer
  Tempo         : 60 BPM opening, gradual acceleration to 90 BPM by end
  Scale/Raga    : Raga Bhairavi (devotional raga suitable for temple worship, evokes devotion and surrender)
  Energy Curve  : low (2/10) meditative opening -> medium (5/10) building communal energy -> moderate-high (7/10) celebratory resolution
  Instruments   : nadaswaram (auspicious temple wind instrument f

#### Step 3: Music Generation

In [13]:
music_gen = SimpleTextToMusicGenerator(model_name=MUSIC_MODEL)

if USE_BEDROCK and "genre" in bgm_prompt:
    bgm_gen = BedrockBGMPromptGenerator()
    music_prompt = bgm_gen.convert_to_music_gen_prompt(bgm_prompt)
elif "generated_prompt" in bgm_prompt:
    music_prompt = bgm_prompt["generated_prompt"]
else:
    music_prompt = str(bgm_prompt)

duration = bgm_prompt.get("duration", bgm_prompt.get("duration_seconds", 30))
print(f"Prompt: {music_prompt}")
print(f"Duration: {duration}s")

Loading model: facebook/musicgen-small
Using device: cuda
Loading processor...
Loading model...


Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

MusicgenForConditionalGeneration LOAD REPORT from: facebook/musicgen-small
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decoder.model.decoder.embed_positions.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ facebook/musicgen-small loaded! Sampling rate: 32000 Hz

Prompt: {'video_context': {'language': 'English', 'content_type': 'devotional', 'setting': 'temple', 'narrative_tone': 'devotional'}, 'emotion_arc': {'opening': 'reverent and contemplative (shrine scene with devotee)', 'development': 'building devotional energy (colorful saris, communal gathering)', 'resolution': 'celebratory devotion (group unity in worship)'}, 'music_blueprint': {'duration_seconds': 15.0, 'tempo_bpm': '60 BPM opening, gradual acceleration to 90 BPM by end', 'musical_scale': 'Raga Bhairavi (devotional raga suitable for temple worship, evokes devotion and surrender)', 'genre': 'South Indian Carnatic devotional with folk percussion layer', 'energy_curve': 'low (2/10) meditative opening -> medium (5/10) building communal energy -> moderate-high (7/10) celebratory resolution', 'primary_instruments': ['nadaswaram (auspicious temple wind instrument for sacred opening)', 'mridangam (rhythmic foundation building from 

In [14]:
audio = music_gen.generate(music_prompt, duration=duration)

bgm_audio_path = f"{OUTPUT_DIR}/{video_name}_bgm.wav"
music_gen.save_audio(audio, bgm_audio_path)
print(f"BGM saved: {bgm_audio_path}")

Token indices sequence length is longer than the specified maximum sequence length for this model (929 > 512). Running this sequence through the model will result in indexing errors


Generating music...
Prompt: {'video_context': {'language': 'English', 'content_type': 'devotional', 'setting': 'temple', 'narrative_tone': 'devotional'}, 'emotion_arc': {'opening': 'reverent and contemplative (shrine scene with devotee)', 'development': 'building devotional energy (colorful saris, communal gathering)', 'resolution': 'celebratory devotion (group unity in worship)'}, 'music_blueprint': {'duration_seconds': 15.0, 'tempo_bpm': '60 BPM opening, gradual acceleration to 90 BPM by end', 'musical_scale': 'Raga Bhairavi (devotional raga suitable for temple worship, evokes devotion and surrender)', 'genre': 'South Indian Carnatic devotional with folk percussion layer', 'energy_curve': 'low (2/10) meditative opening -> medium (5/10) building communal energy -> moderate-high (7/10) celebratory resolution', 'primary_instruments': ['nadaswaram (auspicious temple wind instrument for sacred opening)', 'mridangam (rhythmic foundation building from sparse to active)'], 'secondary_instrum

#### Step 4: Audio Mixing

In [15]:
mixer = VideoMusicPipeline.__new__(VideoMusicPipeline)
mixed_audio_path = f"{OUTPUT_DIR}/{video_name}_mixed_audio.wav"

success = mixer.mix_audio(VIDEO_PATH, bgm_audio_path, mixed_audio_path, bgm_volume=BGM_VOLUME)
if success:
    print(f"Mixed audio: {mixed_audio_path}")
else:
    print("Audio mixing failed - check ffmpeg")

AUDIO PROCESSING
✓ Video has audio - mixing (voice=1.0, bgm=0.4)
✓ Output: /kaggle/working/output/Tamil culture wedding in srilanka _mixed_audio.wav

Mixed audio: /kaggle/working/output/Tamil culture wedding in srilanka _mixed_audio.wav


#### Step 5: Create Final Video

In [16]:
final_video_path = f"{OUTPUT_DIR}/{video_name}_with_bgm.mp4"

success = mixer.create_final_video(VIDEO_PATH, mixed_audio_path, final_video_path)
if success:
    print(f"Final video: {final_video_path}")
else:
    print("Video creation failed - check ffmpeg")

CREATING FINAL VIDEO
✓ Final video: /kaggle/working/output/Tamil culture wedding in srilanka _with_bgm.mp4

Final video: /kaggle/working/output/Tamil culture wedding in srilanka _with_bgm.mp4


In [ ]:
!pip install -q gradio
import gradio as gr

WORK_DIR_UI = "/kaggle/working/gradio_output"
os.makedirs(WORK_DIR_UI, exist_ok=True)

_model_cache = {}

def run_full_pipeline(video_file, bgm_volume, progress=gr.Progress()):
    if video_file is None:
        raise gr.Error("Please upload a video file.")

    vname = Path(video_file).stem

    progress(0.05, desc="\u2728 Loading analysis models...")
    if "analyzer" not in _model_cache:
        _model_cache["analyzer"] = CompleteVideoAnalyzer(whisper_model="base")
    analyzer = _model_cache["analyzer"]

    progress(0.10, desc="\U0001f3a5 Analyzing video (speech + visuals)...")
    analysis = analyzer.analyze_video(video_file, fps=1)
    analysis = convert_to_native(analysis)
    for frame in analysis.get("frame_by_frame_analysis", []):
        frame.pop("image", None)

    duration = analysis["summary"]["duration_seconds"]

    progress(0.35, desc="\U0001f3ad Detecting celebrities & music directors...")
    celebrity_info = None
    try:
        if "celeb_analyzer" not in _model_cache:
            _model_cache["celeb_analyzer"] = CelebrityMusicDirectorAnalyzer()
        celebrity_info = _model_cache["celeb_analyzer"].analyze_celebrities(analysis)
    except Exception as e:
        print(f"Celebrity analysis skipped: {e}")

    progress(0.40, desc="\U0001f9e0 Generating music blueprint via AI...")
    if "bgm_gen" not in _model_cache:
        _model_cache["bgm_gen"] = BedrockBGMPromptGenerator()
    bgm_gen = _model_cache["bgm_gen"]
    blueprint = bgm_gen.generate_prompt(analysis, celebrity_info=celebrity_info)

    progress(0.50, desc="\U0001f3bc Crafting MusicGen prompt...")
    music_prompt = bgm_gen.convert_to_music_gen_prompt(blueprint)

    progress(0.55, desc=f"\U0001f3b5 Generating {duration:.0f}s of music...")
    if "music_gen" not in _model_cache:
        _model_cache["music_gen"] = SimpleTextToMusicGenerator(model_name="facebook/musicgen-small")
    music_gen = _model_cache["music_gen"]
    audio = music_gen.generate(music_prompt, duration=float(duration))
    bgm_path = os.path.join(WORK_DIR_UI, f"{vname}_bgm.wav")
    music_gen.save_audio(audio, bgm_path)

    progress(0.80, desc="\U0001f3a7 Mixing audio tracks...")
    mixer = VideoMusicPipeline.__new__(VideoMusicPipeline)
    mixed_path = os.path.join(WORK_DIR_UI, f"{vname}_mixed.wav")
    ok = mixer.mix_audio(video_file, bgm_path, mixed_path, bgm_volume=float(bgm_volume))
    if not ok:
        raise gr.Error("Audio mixing failed.")

    progress(0.90, desc="\U0001f3ac Rendering final video...")
    final_path = os.path.join(WORK_DIR_UI, f"{vname}_with_bgm.mp4")
    ok = mixer.create_final_video(video_file, mixed_path, final_path)
    if not ok:
        raise gr.Error("Video rendering failed.")

    s = analysis["summary"]
    t = analysis["transcription"]
    mb = blueprint.get("music_blueprint", {})
    arc = blueprint.get("emotion_arc", {})

    # Build celebrity section for summary
    celeb_section = ""
    if celebrity_info and celebrity_info.get("preferred_music_directors"):
        celeb_section = "\nPreferred Music Directors (from celebrity analysis)\n"
        for md in celebrity_info.get("preferred_music_directors", [])[:3]:
            celeb_section += f"  {md['name']}: {md.get('style_description', 'N/A')}\n"
        if celebrity_info.get("detected_celebrities"):
            celeb_section += "Detected Celebrities\n"
            for c in celebrity_info["detected_celebrities"][:3]:
                celeb_section += f"  {c['name']} ({c.get('role','unknown')})\n"
        celeb_section += f"  {celebrity_info.get('filmography_summary', '')}\n"

    summary = (
        f"Video Analysis\n"
        f"  Duration: {s['duration_seconds']}s  |  Frames: {s['total_frames_analyzed']}\n"
        f"  Language: {t.get('language','N/A')}  |  Pacing: {s['motion_analysis']['pacing']}\n"
        f"  Color temp: {s['color_analysis']['dominant_temperature']}\n"
        f"  Objects: {', '.join(o['object'] for o in s['common_objects'][:5])}\n"
        f"  Emotions: {', '.join(e['emotion'] for e in s['detected_emotions'][:3])}\n\n"
        f"Emotion Arc\n"
        f"  {arc.get('opening','--')} > {arc.get('development','--')} > {arc.get('resolution','--')}\n\n"
        f"Music Direction\n"
        f"  Genre: {mb.get('genre', 'N/A')}\n"
        f"  Scale/Raga: {mb.get('musical_scale', 'N/A')}\n"
        f"  Tempo: {mb.get('tempo_bpm', 'N/A')}\n"
        f"  Instruments: {', '.join(mb.get('primary_instruments', []))}\n\n"
        f"MusicGen Prompt\n"
        f"  {music_prompt}"
        f"{celeb_section}"
    )

    progress(1.0, desc="Done!")
    return final_path, bgm_path, summary, final_path


CUSTOM_CSS = """
/* -- GLOBAL -- */
.gradio-container {
    max-width: 100vw !important; width: 100vw !important;
    padding: 0 32px !important;
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important;
    background: url('https://img.freepik.com/premium-vector/colorful-smooth-motion-vector-gradient-wallpaper-design_901408-18501.jpg') no-repeat center center fixed !important;
    background-size: cover !important;
}
.contain { max-width: 100% !important; }
#component-0 { max-width: 100% !important; }

/* -- NAVBAR -- */
.navbar {
    position: sticky; top: 0; z-index: 9999;
    display: flex; align-items: center; justify-content: space-between;
    padding: 14px 36px;
    background: rgba(255,255,255,0.85);
    backdrop-filter: blur(20px); -webkit-backdrop-filter: blur(20px);
    border-bottom: 1px solid #e2e5ea;
    margin: -16px -32px 28px -32px;
    box-shadow: 0 1px 4px rgba(0,0,0,0.03);
}
.navbar-brand { display: flex; align-items: center; gap: 10px; }
.navbar-brand .logo-text {
    font-size: 1.5em; font-weight: 800;
    background: linear-gradient(135deg, #4f46e5, #7c3aed);
    -webkit-background-clip: text; -webkit-text-fill-color: transparent;
}
.navbar-brand .logo-sub {
    font-size: 0.68em; color: #94a3b8; letter-spacing: 2px; text-transform: uppercase;
}
.navbar-links { display: flex; gap: 8px; }
.nav-btn {
    background: transparent; border: 1.5px solid #e2e5ea; color: #64748b;
    padding: 8px 24px; border-radius: 10px; cursor: pointer;
    font-size: 0.88em; font-weight: 600; transition: all 0.2s;
    font-family: 'Inter', sans-serif;
}
.nav-btn:hover { background: #4f46e5; color: white; border-color: #4f46e5; }
.nav-btn.active { background: #eef2ff; color: #4f46e5; border-color: #c7d2fe; }

/* -- ABOUT MODAL -- */
.about-overlay {
    display: none; position: fixed; inset: 0; z-index: 10000;
    background: rgba(15,23,42,0.3); backdrop-filter: blur(10px);
    justify-content: center; align-items: center;
}
.about-overlay.show { display: flex; }
.about-card {
    background: white; border: 1px solid #e2e5ea; border-radius: 24px;
    padding: 44px 48px; max-width: 620px; width: 92%;
    color: #1e293b; position: relative;
    box-shadow: 0 25px 60px rgba(0,0,0,0.1);
}
.about-card h2 {
    margin: 0 0 4px; font-size: 1.7em; font-weight: 800;
    background: linear-gradient(135deg, #4f46e5, #7c3aed);
    -webkit-background-clip: text; -webkit-text-fill-color: transparent;
}
.about-card .tagline { color: #94a3b8; font-size: 0.95em; margin-bottom: 20px; }
.about-card p { color: #64748b; line-height: 1.75; font-size: 0.93em; margin: 12px 0; }
.about-card .tech-stack { display: flex; flex-wrap: wrap; gap: 8px; margin-top: 18px; }
.about-card .tech-tag {
    background: #f1f5f9; color: #4f46e5; border: 1px solid #e2e8f0;
    padding: 6px 14px; border-radius: 20px; font-size: 0.8em; font-weight: 600;
}
.about-close {
    position: absolute; top: 18px; right: 22px;
    background: #f1f5f9; border: none; color: #94a3b8; font-size: 1.3em;
    cursor: pointer; width: 36px; height: 36px; border-radius: 50%;
    display: flex; align-items: center; justify-content: center; transition: all 0.2s;
}
.about-close:hover { background: #4f46e5; color: white; }

/* -- HERO -- */
.hero-section {
    text-align: center; padding: 52px 28px 32px;
    background: linear-gradient(135deg, #eef2ff, #f5f3ff, #faf5ff);
    border-radius: 24px; margin-bottom: 32px;
    border: 1px solid #e2e5ea;
    position: relative; overflow: hidden;
}
.hero-disc {
    position: absolute; left: 32px; top: 50%; transform: translateY(-50%);
    width: 90px; height: 90px; border-radius: 50%;
    object-fit: cover;
    box-shadow: 0 4px 20px rgba(79,70,229,0.25);
    animation: discFloat 3s ease-in-out infinite;
}
@keyframes discFloat { 0%,100%{transform:translateY(-50%)} 50%{transform:translateY(calc(-50% - 14px))} }
.hero-section h1 {
    font-size: 2.6em; margin: 0; color: #1e293b; font-weight: 800;
    letter-spacing: -0.5px;
}
.hero-section .subtitle { color: #64748b; font-size: 1.12em; margin-top: 12px; }
@keyframes float { 0%,100%{transform:translateY(0)} 50%{transform:translateY(-8px)} }
@keyframes shimmer { 0%{background-position:0% center} 100%{background-position:200% center} }
.algo-bounce {
    display: inline-block; margin-top: 20px;
    font-size: 0.95em; font-weight: 700; letter-spacing: 1.5px;
    background: linear-gradient(90deg, #4f46e5, #7c3aed, #a78bfa, #4f46e5);
    background-size: 200% auto;
    -webkit-background-clip: text; -webkit-text-fill-color: transparent;
    animation: float 3s ease-in-out infinite, shimmer 4s linear infinite;
}

/* -- CARDS -- */
.upload-card, .control-panel {
    background: white; border-radius: 20px; padding: 28px;
    border: 1px solid #e2e5ea;
    box-shadow: 0 1px 3px rgba(0,0,0,0.03), 0 4px 12px rgba(0,0,0,0.02);
}
.step-badge {
    display: inline-block; background: #eef2ff;
    color: #4f46e5; padding: 6px 18px; border-radius: 20px;
    font-size: 0.82em; font-weight: 700; margin-bottom: 12px;
    border: 1px solid #c7d2fe; letter-spacing: 0.5px;
}

/* -- GENERATE BUTTON -- */
.generate-btn {
    background: linear-gradient(135deg, #4f46e5, #7c3aed) !important;
    color: white !important; font-weight: 700 !important; font-size: 1.08em !important;
    border: none !important; border-radius: 14px !important; padding: 16px 32px !important;
    transition: all 0.25s !important; width: 100% !important; margin-top: 16px !important;
    box-shadow: 0 4px 16px rgba(79,70,229,0.25) !important;
    letter-spacing: 0.3px !important;
}
.generate-btn:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 30px rgba(79,70,229,0.4) !important;
}

/* -- RESULTS -- */
.results-section {
    background: white; border-radius: 20px; padding: 28px;
    border: 1px solid #e2e5ea; margin-top: 20px;
    box-shadow: 0 1px 3px rgba(0,0,0,0.03), 0 4px 12px rgba(0,0,0,0.02);
}
.summary-box textarea {
    background: #f8fafc !important; color: #334155 !important;
    border: 1px solid #e2e8f0 !important; border-radius: 14px !important;
    font-family: 'JetBrains Mono', 'Fira Code', 'Cascadia Code', monospace !important;
    font-size: 0.87em !important; line-height: 1.75 !important;
    padding: 16px !important;
}
.pipeline-info {
    margin-top: 20px; padding: 14px 18px;
    background: #f8fafc; border-radius: 14px; border: 1px solid #e2e8f0;
}
.pipeline-info p { color: #64748b; font-size: 0.83em; margin: 0; line-height: 1.6; }
.pipeline-info b { color: #4f46e5; }

/* -- HIDE GRADIO BRANDING -- */
footer { display: none !important; }
.built-with, .show-api, .gr-button.icon-button { display: none !important; }
a[href*='huggingface'], img[src*='huggingface'], img[alt*='Hugging Face'] { display: none !important; }
"""

with gr.Blocks(
    title="AlgoRhythm — Video BGM Generator",
    theme=gr.themes.Base(
        primary_hue=gr.themes.colors.indigo,
        secondary_hue=gr.themes.colors.violet,
        neutral_hue=gr.themes.colors.gray,
        font=gr.themes.GoogleFont("Inter"),
    ).set(
        body_background_fill="transparent",
        body_text_color="#1e293b",
        block_background_fill="white",
        block_border_color="#e2e5ea",
        block_label_text_color="#64748b",
        input_background_fill="#f8fafc",
        button_primary_background_fill="#4f46e5",
        button_primary_text_color="white",
    ),
    css=CUSTOM_CSS,
) as demo:

    gr.HTML("""
    <nav class="navbar">
        <div class="navbar-brand">
            <div>
                <div class="logo-text">\U0001f3b5 AlgoRhythm</div>
                <div class="logo-sub">AI Music Engine</div>
            </div>
        </div>
        <div class="navbar-links">
            <button class="nav-btn active" onclick="document.getElementById('aboutModal').classList.remove('show')">Home</button>
            <button class="nav-btn" onclick="document.getElementById('aboutModal').classList.add('show')">About</button>
        </div>
    </nav>

    <div id="aboutModal" class="about-overlay" onclick="if(event.target===this)this.classList.remove('show')">
        <div class="about-card">
            <button class="about-close" onclick="document.getElementById('aboutModal').classList.remove('show')">&times;</button>
            <h2>AlgoRhythm</h2>
            <div class="tagline">AI-Powered Culturally-Aware Background Music</div>
            <p>AlgoRhythm analyzes your video frame-by-frame \u2014 detecting objects, emotions, speech, colors, and motion \u2014 then uses AI to compose a culturally-aware music blueprint and generate original background music that fits perfectly.</p>
            <p>The pipeline preserves your original audio (dialogue, narration) and mixes the generated BGM at your chosen volume level.</p>
            <div class="tech-stack">
                <span class="tech-tag">Whisper</span>
                <span class="tech-tag">CLIP</span>
                <span class="tech-tag">BLIP</span>
                <span class="tech-tag">DETR</span>
                <span class="tech-tag">Claude (Bedrock)</span>
                <span class="tech-tag">MusicGen</span>
                <span class="tech-tag">FFmpeg</span>
            </div>
        </div>
    </div>

    <div class="hero-section">
        <img class="hero-disc" src="https://img.freepik.com/premium-photo/intricate-harmonies-black-background-music-notes-circle-design_983420-153591.jpg" alt="Music Disc" />
        <h1>\U0001f3ac Video BGM Generator</h1>
        <p class="subtitle">Upload a video, adjust volume, hit Generate \u2014 AI handles the rest</p>
        <div class="algo-bounce">AlgoRhythm \u2014 Try the Rhythm for your Video</div>
    </div>
    """)

    with gr.Row(equal_height=True):
        with gr.Column(scale=3, elem_classes=["upload-card"]):
            gr.HTML('<span class="step-badge">UPLOAD</span>')              # ✅ FIXED
            video_input = gr.Video(label="Drop your video here", height=360)

        with gr.Column(scale=2, elem_classes=["control-panel"]):
            gr.HTML('<span class="step-badge">SETTINGS</span>')            # ✅ ALREADY CORRECT
            gr.Markdown("Control how loud the AI-generated music sits behind your original audio.")
            bgm_vol_slider = gr.Slider(
                0.05, 1.0, value=0.40, step=0.05,
                label="BGM Volume",
                info="0.05 = barely audible, 1.0 = full volume"
            )
            gr.HTML("<div style='height:16px'></div>")
            btn_go = gr.Button(
                "Generate Background Music",
                variant="primary",
                size="lg",
                elem_classes=["generate-btn"]
            )
            gr.HTML("""
            <div class="pipeline-info">
                <p><b>Pipeline:</b> Video Analysis \u2192 AI Music Blueprint \u2192 MusicGen \u2192 Audio Mixing \u2192 Final Render</p>
            </div>
            """)

    gr.HTML('<div style="margin-top:24px"><span class="step-badge">RESULTS</span></div>')  # ✅ FIXED

    with gr.Row(elem_classes=["results-section"]):
        with gr.Column(scale=3):
            final_video = gr.Video(label="Final Video with BGM", height=420)
        with gr.Column(scale=2):
            bgm_audio = gr.Audio(label="Generated BGM Preview", type="filepath")
            download_file = gr.File(label="Download Final Video")

    summary_box = gr.Textbox(
        label="Pipeline Report",
        lines=14,
        interactive=False,
        elem_classes=["summary-box"]
    )

    btn_go.click(
        fn=run_full_pipeline,
        inputs=[video_input, bgm_vol_slider],
        outputs=[final_video, bgm_audio, summary_box, download_file]
    )

demo.queue()
demo.launch(share=True, debug=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://ebc94746e8db0a5fc0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


INITIALIZING COMPLETE VIDEO ANALYZER
Using device: cuda

Loading Whisper model...
✓ Whisper loaded

Loading CLIP model...
Failed with openai/clip-vit-base-patch32: CLIPModel.__init__() got an unexpected keyword argument 'resume_download'
Trying alternative: openai/clip-vit-base-patch16...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ CLIP loaded

Loading BLIP captioning model...


...

100%|██████████| 543/543 [00:00<00:00, 1910.59frames/s]


✓ Detected language: en
STEP 2: FRAME EXTRACTION
Video: 5.0s, 29.2 FPS, extracting at 1 FPS
✓ Extracted 5 frames

STEP 3: FRAME-BY-FRAME ANALYSIS
Analyzing frame 1/5 at 0.00s...
Analyzing frame 2/5 at 0.99s...
Analyzing frame 3/5 at 1.98s...
Analyzing frame 4/5 at 2.98s...
Analyzing frame 5/5 at 3.97s...
✓ Frame analysis complete

STEP 4: GENERATING SUMMARY
✓ Summary generated

Initializing Bedrock client with API Key...
Connected to Endpoint : https://bedrock-runtime.us-east-1.amazonaws.com/model/arn%3Aaws%3Abedrock%3Aus-east-1%3A926251048803%3Aapplication-inference-profile%2Faw4utpbetipp/invoke
Model                 : arn:aws:bedrock:us-east-1:926251048803:application-inference-profile/aw4utpbetipp

LLM CALL #1 - VIDEO ANALYSIS -> MUSIC BLUEPRINT (JSON)
  Video duration : 4.0s
  Language       : en
  Sending to Bedrock...

Music Blueprint Generated!
  Content Type  : narrative
  Language      : English
  Genre         : Contemporary dramatic underscore | Psychological tension music
 

Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

MusicgenForConditionalGeneration LOAD REPORT from: facebook/musicgen-small
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decoder.model.decoder.embed_positions.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ facebook/musicgen-small loaded! Sampling rate: 32000 Hz

Generating music...
Prompt: Western contemporary film score, psychological tension underscore, D minor with chromatic dissonance, 4/4 irregular accents with heartbeat pulse, sad grief opening -> rising anxiety and anger -> unresolved fear, slow 70 BPM deliberate, low cello sustained notes, sparse piano single strikes, subtle viola tremolo texture, distant ambient pad drone, minimal sparse intimate, brooding tense unresolved, modern cinematic hybrid production, no vocals instrumental only
Duration: 4.0s
Generating 200 tokens...
✓ Generation complete!

✓ Audio saved to: /kaggle/working/gradio_output/MicrosoftTeams-video_bgm.wav (3.94s)
AUDIO PROCESSING
✓ Video has audio - mixing (voice=1.0, bgm=0.4)
✓ Output: /kaggle/working/gradio_output/MicrosoftTeams-video_mixed.wav

CREATING FINAL VIDEO
✓ Final video: /kaggle/working/gradio_output/MicrosoftTeams-video_with_bgm.mp4

